### Analyze + Plot Eye Saccades
The goal of this script is to determine wether the animal's saccades
are aligned with the location of the stimulus. In other words, if the
animal is looking at the salient part of the screen. The script:

  - Requires a path to a folder on the lab's server (X:\)
  - Loads right- and left-eye Bonsai CSV outputs, IMU & stimulus TTL files.
  - Computes eye-centric gaze, detects saccades (optionally removing blinks).
  - Creates three kinds of figures and writes them to  <folder_path>/Results/ :
      ─ “ALL”  : quiver of every saccade  + polar & linear angle histograms
      ─ LR/UD  : same layout but restricted to each stimulus direction
  - Global X/Y limits are derived from the full session; every quiver plot
    shares them so comparisons are direct.
  - Figure filenames:  <session>_<Eye>_<condition>.png
      e.g.  Tsh001_2025-06-11T13_02_29_Right_ALL_Interleaved.png


### How saccades are detected in this script

1. **Eye-centred coordinates**  
   Subtract the mid-point of the two eye-corner markers so gaze is relative to the eye, not the camera.

2. **Denoise (`medfilt_vec`)**  
   A 3-point median filter removes single-frame tracking jitter.

3. **Pixel → degree conversion**  
   Divide by the calibration factor ( ≈ 3.76 px = 1 °) to work in visual degrees.

4. **Instantaneous velocity (`frame_velocity`)**  
   Take the frame-to-frame difference in x and y; combine into a scalar speed (deg / frame).

5. **Speed threshold**  
   If speed ≥ `1.5 deg/frame`, mark that frame as a saccade (`saccade_indices`).

6. **Blink removal (`blink_frames`, optional)**  
   When VD-axis files are present, large jumps in eyelid separation flag blinks; those frames are deleted from `saccade_indices`.

The cleaned list **`saccade_indices`** feeds every quiver plot, PCA arrow, and angle histogram in the notebook.




Tested with Python 3.12 (Conda env “EyeHeadCoupling”).

  Author:  <Ratnadeep Pal @SarvestaniLab>   –  Last update: 2025-06-15

## Import the required libraries


In [4]:

import sys
import os
import numpy as np
import pandas as pd
from scipy.fft import fft
from scipy.signal import ShortTimeFFT,butter,hilbert,sosfiltfilt,medfilt,filtfilt
from scipy.signal.windows import gaussian
import scipy.stats as stats
from scipy.spatial import ConvexHull
from scipy.interpolate import interp1d
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import tkinter as tk
from tkinter import filedialog
from sklearn.decomposition import PCA
import os
from io import StringIO
from matplotlib import gridspec
from pathlib import Path
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec
import re
from datetime import datetime
from matplotlib.patches import FancyArrowPatch
from itertools import cycle
from matplotlib import cm
from matplotlib.collections import LineCollection
import matplotlib
import cv2



%matplotlib qt

_STYLE_PATH ="C:\\Users\\rp672\\Documents\\EyeHeadCoupling\\Python\\style.mplstyle"       
plt.style.use(_STYLE_PATH)

## Define parameters

In [5]:
cal = 3.76  # Calibration factor for the pixels to degrees
ttl_freq = 60  # TTL frequency in Hz

# Parameters for nlink and saccade detection
blink_detection = 1
blink_thresh= 10
saccade_thresh= 1.0
torsion_velocity_thresh = 1.5
saccade_win=0.7    # Window size for saccade detection in seconds

#folder_path = select_folder() #this won't work if you're running jupyter lab in browser, so hard coding


###################################### Paris

#First day when we started doing interleaved stim.
#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-06-09T13_09_02\\" # interleaved

#Second day where she licked a lot!
#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-06-11T12_50_45\\" #no stim
#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-06-11T13_02_29\\" # interleaved
#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-06-11T13_15_39\\" #  interleaved

#Not much licking on this day
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-06-13T13_07_55\\" #interleaved
#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-06-13T13_25_04\\" #no-stim session

#Motivated on this day, but first day where juice was only given for saccades
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-06-16T12_13_51\\" #interleaved

#second day juice was given for saccades only but she was stressed 
#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-06-17T12_50_28\\" #interleaved

#third day juce was given for saccades only, but she was not thirsty
#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-06-18T12_19_39\\" #interleaved

#fourth day, but she didn't pay attention the whole time
#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-06-19T12_17_52\\" #interleaved


folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-06-30T15_20_56\\" #just L/R


#this is after ratnadeep fixed a bunch of issues with the code! and data looks great!!
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-02T15_14_29\\" #just L/R


#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-07T14_49_12\\" #just L/R
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-07T15_12_22\\" #just L/R

#good session!
#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-08T15_04_12\\" #just L/R

#good session! first day we tried U/D stim
#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-09T15_29_43\\" #just L/R
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-09T15_49_30\\" #L/R and U/D 

#tough day, she was stressed and didn't pay attention
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-10T15_36_39\\" # U/D --bad
#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-10T15_47_04\\" #L/R and U/D 
######################################## Bayleaf
#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\Rat22_Bayleaf_server\Rat022_2025-06-10T16_23_02\\"   #no stim
#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\Rat22_Bayleaf_server\Rat022_2025-06-10T16_10_21\\"   #no stim

# pretty bad performance this day, used wrong head bar
#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-11T14_22_34\\" #L/R and U/D 
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-11T14_49_16\\" #L/R and U/D 


#came in on a sunday to do a session, figuring out she'd be motivated and calm
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-13T17_38_53\\" #L/R and U/D 
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-13T17_46_15\\" #L/R and U/D, moved u/down closer
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-13T17_52_47\\" #U/D only
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-13T18_19_05\\" #L/R only

#good long session today
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-14T14_58_27\\" #L/R only
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-14T15_04_46\\" #L/R only
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-14T15_11_17\\" #interleaved
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-14T15_16_32\\" #interleaved
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-14T15_26_38\\" #Up/Down
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-14T15_39_13\\" #Up / 2 levels
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-14T15_49_51\\" #Up / Down 2 levels
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-14T15_56_54\\" #Up / Down 1 levels

#good long session today, where we looked at torsion online
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-15T16_16_03\\" #interleaved
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-15T16_36_23\\" #U/D only
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-15T16_45_37\\" #L/R


################################################### #torsion training
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-17T15_32_42\\" #interleaved


folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-21T15_08_33\\" #Up/down
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-21T15_45_48\\" #Up/down
#testing monitor positions to see if down torsion gets better
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-22T15_02_57\\" #Up/down
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-22T15_25_56\\" #Up/down

##################################################### fixation training ####################
# #first day where we started punishing for not fixating during blue spot
#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-30T15_29_31\\" #interleaved
# folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-30T15_39_26\\" #interleaved
# folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-30T15_47_15\\" #interleaved

# # #second day where we punish for not fixating during blue spot
#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-31T15_23_59\\" #interleaved
#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-07-31T15_56_37\\" #interleaved

#third day where we punish for not fixating during blue spot
#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-08-01T15_15_48\\" #interleaved
folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-08-06T16_40_40\\" #interleaved


###################################################################### anti-saccade training
#folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-08-11T16_40_24\\" 
# folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-08-12T15_04_14\\"
# folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-08-13T14_48_16\\" #interleaved
# folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-08-13T15_02_30\\" #interleaved
# folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-08-14T15_06_12\\" #interleaved
# folder_path = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-08-14T15_12_24\\" #interleaved



results_dir = Path(folder_path) / "Results\\"
results_dir.mkdir(exist_ok=True)

session_name = os.path.basename(folder_path.rstrip("/\\"))




## Define functions


In [6]:

def get_session_date_from_path(path):
    match = re.search(r"\d{4}-\d{2}-\d{2}", path)
    if match:
        return datetime.strptime(match.group(), "%Y-%m-%d")
    else:
        raise ValueError("No valid date (YYYY-MM-DD) found in path")
    
def determine_camera_side(path, cutoff_date_str="2025-06-30"):
    session_date = get_session_date_from_path(path)
    cutoff_date = datetime.strptime(cutoff_date_str, "%Y-%m-%d")
    return "L" if session_date >= cutoff_date else "R"


# Prompt the user to select a folder
def select_folder():
    root = tk.Tk()
    root.withdraw()  # Hide the main window
    directory = filedialog.askdirectory()  # Open the file selection dialog
    return directory

# Prompt the user to open a file 
def select_file():
    root = tk.Tk()
    root.withdraw()  # Hide the main window
    file_path = filedialog.askopenfilename()  # Open the file selection dialog
    return file_path

def choose_option(option1, option2, option3, option4):
    result = {}

    def select(choice):
        result['value'] = choice
        root.destroy()

    root = tk.Tk()
    root.title("Choose the type of visual stim")

    tk.Label(root, text="Please choose the type of visual stim:").pack(pady=10)
    tk.Button(root, text=option1, width=12, command=lambda: select(option1)).pack(side='left', padx=10, pady=10)
    tk.Button(root, text=option2, width=12, command=lambda: select(option2)).pack(side='left', padx=10, pady=10)
    tk.Button(root, text=option3, width=12, command=lambda: select(option3)).pack(side='left', padx=10, pady=10)
    tk.Button(root, text=option4, width=12, command=lambda: select(option4)).pack(side='left', padx=10, pady=10)

    # Manual event loop, blocks until window is destroyed
    while not result.get('value'):
        root.update()

    return result['value']

# Prompt the user to choose the type of visual stim
#stim_type = choose_option("None","LR","UD","Interleaved")


# Function to remove parentheses characters from a line
def remove_parentheses_chars(line):
    # Remove only '(' and ')' characters
    return line.replace('(', '').replace(')', '').replace('True', '1').replace('False', '0')
def clean_csv(filename):
    with open(filename, 'r') as f:
        lines = [remove_parentheses_chars(line) for line in f]
    # Join lines and create a file-like object
        cleaned = StringIO(''.join(lines))
        return cleaned

# Butterworth filter to remove high frequency noise
def butter_noncausal(signal, fs, cutoff_freq=1, order=4):
    sos = butter(order, cutoff_freq/(fs/2), btype='low', output='sos')  # 50 Hz cutoff frequency
    return sosfiltfilt(sos, signal)   

def interpolate_nans(arr):
    nans = np.isnan(arr)
    x = np.arange(len(arr))
    arr[nans] = np.interp(x[nans], x[~nans], arr[~nans])
    return arr

def rotation_matrix(angle_rad):
    return np.array([[np.cos(angle_rad), -np.sin(angle_rad)],
                     [np.sin(angle_rad), np.cos(angle_rad)]])


pca = PCA(n_components=2)

def vector_to_rgb(angle, absolute): ##Got it from https://stackoverflow.com/questions/19576495/color-matplotlib-quiver-field-according-to-magnitude-and-direction
    global max_abs

    # normalize angle
    angle = angle % (2 * np.pi)
    if angle < 0:
        angle += 2 * np.pi

    # return matplotlib.colors.hsv_to_rgb((angle / 2 / np.pi, 
    #                                      absolute / max_abs, 
    #                                      absolute / max_abs))
    return matplotlib.colors.hsv_to_rgb((angle / 2 / np.pi, 
                                         1, 
                                         1))


def plot_angle_distribution(angle, ax_polar, num_bins=18):
    """
    Plots a normalized polar histogram of angles.

    Parameters:
        angle (np.ndarray): array of saccade angles in radians
        ax_polar (matplotlib.axes._subplots.PolarAxesSubplot): the polar subplot to draw on
        num_bins (int): number of histogram bins
    """
    angle_2pi = np.where(angle < 0, angle + 2 * np.pi, angle)
    counts, bin_edges = np.histogram(angle_2pi, bins=num_bins, range=(0, 2 * np.pi))
    counts = counts / np.size(angle_2pi)  # Normalize
    width = np.diff(bin_edges)

    bars = ax_polar.bar(bin_edges[:-1], counts, width=width, align='edge', color='b', alpha=0.5, edgecolor='k')
    ax_polar.set_title("Normalized angle distribution")
    ax_polar.set_yticklabels([])

def plot_linear_histogram(angles, ax, num_bins=18):
    ang_deg = np.degrees(angles)
    ang_deg = np.mod(ang_deg, 360)
    counts, bins = np.histogram(ang_deg, bins=num_bins, range=(0, 360))
    counts = counts / ang_deg.size
    ax.bar(bins[:-1], counts, width=np.diff(bins), color="b", alpha=0.5, edgecolor="k")
    ax.set_xlabel("Angle (deg)")
    ax.set_ylabel("Normalised count")
    ax.set_title("Linear angle histogram")    

def detect_saccades(
    marker1_x, marker1_y, marker2_x, marker2_y,
    gaze_x, gaze_y,
    eye_frames,
    calibration_factor,
    blink_velocity_threshold, 
    saccade_threshold,
    blink_detection=0,
    vd_axis_lx=None, vd_axis_ly=None, vd_axis_rx=None, vd_axis_ry=None,
    torsion_angle=None,
    saccade_threshold_torsion=None,  
):
    ################# Analyze eye saccades function #################
    """
    #  Compute eye position in eye-centered coordinates.
    # Filter and convert raw gaze to degrees using calibration factor.
    # Compute eye position velocity to detect saccades.
    # Identify saccades that exceed a velocity threshold.
    # (Optional) Remove saccades likely caused by blinks.
    """
    

    # 1. eye-centred coordinates  →  degrees
    eye_origin = np.column_stack(((marker1_x + marker2_x) / 2.0,
                                (marker1_y + marker2_y) / 2.0))
    eye_camera = np.column_stack((gaze_x - eye_origin[:, 0],
                                gaze_y - eye_origin[:, 1])).astype(np.float64, copy=False)
    
    eye_angle = np.arctan2(marker2_y - marker1_y, marker2_x - marker1_x)

    # tiny denoise
    eye_camera[:, 0] = medfilt(eye_camera[:, 0], kernel_size=3)
    eye_camera[:, 1] = medfilt(eye_camera[:, 1], kernel_size=3)

    #read in 1 or 2 calibration factors
    cal = np.asarray(calibration_factor, dtype=np.float64)

    if cal.ndim == 0:                # scalar px/deg (same for X,Y)
        fx = fy = float(cal)
    elif cal.shape == (2,):          # [fx, fy] per-axis px/deg
        fx, fy = float(cal[0]), float(cal[1])

    eye_camera[:, 0] /= fx
    eye_camera[:, 1] /= fy

    
    # 2. instantaneous velocity  →  speed
    dx = np.ediff1d(eye_camera[:, 0], to_begin=0)
    dy = np.ediff1d(eye_camera[:, 1], to_begin=0)
    xy_speed = np.sqrt(dx**2 + dy**2)

    xy_mask = xy_speed >= saccade_threshold

    # 3. eye position velocity  based on torsion
    if torsion_angle is not None:
        torsion_angle = interpolate_nans(torsion_angle)
        dtheta = np.ediff1d(torsion_angle, to_begin=0)
        torsion_speed = np.abs(dtheta)
    else:
        torsion_speed = np.zeros_like(xy_speed)

    # 4. Thresholding based on speed masks
    torsion_mask = torsion_speed >= (saccade_threshold_torsion or np.inf)
    
    #5. Detect saccades based on xy_speed and torsion_speed

    saccade_indices_xy = np.where(xy_mask)[0] # ← row indices (0…7158)
    saccade_frames_xy = eye_frames[saccade_indices_xy] # ← absolute Bonsai frames

    saccade_indices_theta = np.where(torsion_mask)[0] # ← row indices (0…7158)
    saccade_frames_theta = eye_frames[saccade_indices_theta] # ← absolute Bonsai frames

    # 6. Package eye positions and velocity into output (can be 3D if torsion is included)
    if torsion_angle is not None:
        eye_pos = np.column_stack([eye_camera, torsion_angle])
        eye_vel = np.column_stack([dx, dy, dtheta])
    else:
        eye_pos = eye_camera
        eye_vel = np.column_stack([dx, dy])





    # Plot saccade and threshold to make sure it's detected
    fig, (ax, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    frames = np.arange(len(xy_speed))

    # ─── Plot XY (translational) saccades ───
    ax.plot(frames, xy_speed, linewidth=0.8, label='Speed (°/frame)')
    ax.scatter(saccade_indices_xy, xy_speed[saccade_indices_xy],
            color='tab:red', s=12, label='Saccade idx')
    ax.axhline(saccade_threshold, color='tab:orange',
            linestyle='--', label=f'Threshold = {saccade_threshold}')
    ax.set_ylabel('Speed (° / frame)')
    ax.set_title('Instantaneous XY speed with detected saccade frames')
    ax.legend()
    ax.grid(alpha=.3)

    # ─── Plot torsional saccades ───
    ax2.plot(frames, torsion_speed, linewidth=0.8, label='Torsion Speed (°/frame)')
    ax2.scatter(saccade_indices_theta, torsion_speed[saccade_indices_theta],
                color='tab:purple', s=12, label='Torsion idx')
    ax2.axhline(saccade_threshold_torsion, color='tab:purple',
                linestyle='--', label=f'Threshold = {saccade_threshold_torsion}')
    ax2.set_xlabel('Frame number')
    ax2.set_ylabel('Torsion Speed (° / frame)')
    ax2.set_title('Instantaneous torsion speed with detected torsional saccades')
    ax2.legend()
    ax2.grid(alpha=.3)

    plt.tight_layout()
    plt.show()


    # save alongside other figures
    prob_fname = f"{session_name}_saccades.png"
    fig.savefig(results_dir / prob_fname, dpi=300, bbox_inches='tight')


    # 3. optional blink removal
    if blink_detection:
        vd_axis_left = np.vstack([vd_axis_lx, vd_axis_ly]).T
        vd_axis_right = np.vstack([vd_axis_rx, vd_axis_ry]).T
        vd_axis_d = np.linalg.norm(vd_axis_right - vd_axis_left, axis=1)
        vd_axis_vel = np.gradient(vd_axis_d)
        blink_indices = np.where(np.abs(vd_axis_vel) > blink_velocity_threshold)
        saccade_indices_xy = saccade_indices_xy[~np.isin(saccade_indices_xy, blink_indices)]

    return {
        "eye_pos":          eye_pos, #in degrees
        "eye_vel":          eye_vel, #in degrees/second
        "saccade_indices_xy":  saccade_indices_xy,
        "saccade_frames_xy":   saccade_frames_xy,
        "saccade_indices_theta":  saccade_indices_theta,
        "saccade_frames_theta":   saccade_frames_theta,
        }



def organize_stims(
    go_frame, 
    go_dir_x = None,
    go_dir_y = None):
    
    has_lr = go_dir_x is not None and np.any(go_dir_x != 0)
    has_ud = go_dir_y is not None and np.any(go_dir_y != 0)

    direction_sets = {}

    if has_lr:
        direction_sets["Left"]  = go_dir_x < 0
        direction_sets["Right"] = go_dir_x > 0
    if has_ud:
        direction_sets["Down"] = go_dir_y < 0
        direction_sets["Up"]   = go_dir_y > 0
    if not direction_sets:
        direction_sets["All"] = np.full(len(go_frame), True)

    stim_frames = {lab: go_frame[mask] for lab, mask in direction_sets.items()}

    # Return the inferred stim type too
    if has_lr and has_ud:
        stim_type = "Interleaved"
    elif has_lr:
        stim_type = "LR"
    elif has_ud:
        stim_type = "UD"
    else:
        stim_type = "None"

    return stim_frames, stim_type



def sort_plot_saccades(
    #sorts saccades by stimulus onset
    saccades,
    saccade_window,  # seconds before/after each stimulus
    session_path,
    stim_type='None',
    eye_name='Eye',

):

    eye_pos = saccades["eye_pos"]
    eye_pos_diff = saccades["eye_vel"]
    saccade_indices_xy = saccades["saccade_indices_xy"]
    saccade_frames_xy= saccades["saccade_frames_xy"]
    saccade_indices_theta= saccades["saccade_indices_theta"]
    saccade_frames_theta = saccades["saccade_frames_theta"]
    stim_frames = saccades["stim_frames"]
    session_name = os.path.basename(session_path.rstrip("/\\"))

    # ───────── global axis limits (all saccades) ─────────

    if saccade_indices_theta is not None and len(saccade_indices_theta) > 0:
        saccade_indices_theta = np.array(saccade_indices_theta, dtype=int)
        t_all = eye_pos[saccade_indices_theta, 2]
    else:
        t_all = None

    if eye_pos_diff.shape[1] == 3:
        dx, dy, dtheta = eye_pos_diff[:, 0], eye_pos_diff[:, 1], eye_pos_diff[:, 2]
        x_all, y_all = eye_pos[saccade_indices_xy, 0], eye_pos[saccade_indices_xy, 1]
        # Extract torsion angles and convert to degrees
        t_all = eye_pos[saccade_indices_theta,2] if saccade_indices_theta is not None else None
        torsion_present = True
    else:
        dx, dy = eye_pos_diff[:, 0], eye_pos_diff[:, 1]
        x_all, y_all = eye_pos[saccade_indices_xy, 0], eye_pos[saccade_indices_xy, 1]
        t_all = None
        dtheta = None
        saccade_indices_theta = None
        saccade_frames_theta = None
        torsion_present = False


    pad   = 0.10
    rngX  = x_all.max() - x_all.min()
    rngY  = y_all.max() - y_all.min()
    X_LIM = (x_all.min() - pad*rngX, x_all.max() + pad*rngX)
    Y_LIM = (y_all.min() - pad*rngY, y_all.max() + pad*rngY)
    max_abs = np.max(np.hypot(dx[saccade_indices_xy], dy[saccade_indices_xy]))
    
    #calculate angles for all translational saccades
    angle_all = np.arctan2(dy[saccade_indices_xy],
                            dx[saccade_indices_xy])
    n_all = len(saccade_indices_xy)

    # ───────── master figure (ALL saccades) ─────────
    fig = plt.figure(figsize=(11, 6))
    gs  = gridspec.GridSpec(2, 2, width_ratios=[3, 2])
    ax_quiver = fig.add_subplot(gs[:, 0])
    ax_polar  = fig.add_subplot(gs[0, 1], polar=True)
    ax_linear = fig.add_subplot(gs[1, 1])

    ax_quiver.set_xlim(*X_LIM); ax_quiver.set_ylim(*Y_LIM)
    ax_quiver.set_xlabel('X (°)'); ax_quiver.set_ylabel('Y (°)')
    ax_quiver.set_title(
        f"{session_name}\n"
        f"All translational saccades ({n_all}) — {eye_name}  (stim: {stim_type})\n"
        f"saccade_thresh = {saccade_thresh}, saccade_win = {saccade_win}s\n"
        f"blink_thresh = {blink_thresh}, blink_detection = {blink_detection}s\n"
        )

    cols = np.array([vector_to_rgb(a, max_abs) for a in angle_all])
    ax_quiver.quiver(x_all, y_all,
                        dx[saccade_indices_xy],
                        dy[saccade_indices_xy],
                        angles='xy', scale_units='xy', scale=1,
                        color=cols, alpha=.5)

    # PCA arrows (unchanged)
    # pca.fit(eye_pos_diff[saccade_indices] /
    #         np.linalg.norm(eye_pos_diff[saccade_indices], axis=1, keepdims=True))
    # for i, (vec, var) in enumerate(zip(pca.components_, pca.explained_variance_ratio_)):
    #     ax_quiver.arrow(np.mean(x_all), np.mean(y_all),
    #                     *(vec * 10 * np.sqrt(var)),
    #                     color=['k', 'b'][i], width=0.1,
    #                     label=f'PC{i+1} ({var:.2f} var)')
    # ax_quiver.legend()

    plot_angle_distribution(angle_all, ax_polar)
    plot_linear_histogram(angle_all, ax_linear)
    plt.tight_layout()

    # save master figure
    all_fname = f"{session_name}_{eye_name}_ALL_{stim_type}.png"
    fig.savefig(results_dir / all_fname, dpi=300, bbox_inches='tight')


    # Determine the overall frame range [0, last_frame]
    last_frame = int(saccade_frames_xy.max())
    clipped_any = False
    plot_window = np.arange(0,saccade_window,1)

    # ───────── one figure per stimulus label (skip "All") ─────────
    for label, frames in stim_frames.items():
        if label == "All":
            continue

        # gather 1st saccades within ±plot_window around each stim

        idx_buf = []  # buffer to collect saccade indices for this label

        # sort saccade frames to ensure they are in order
        sorted_pairs_xy = sorted(zip(saccade_frames_xy, saccade_indices_xy))

        for f in frames:

            lower_bound = max(f + plot_window[0], 0)
            upper_bound = min(f + plot_window[-1], saccade_frames_xy.max())

            for sf, idx in sorted_pairs_xy:
                if sf < lower_bound:
                    continue
                elif sf <= upper_bound:
                    idx_buf.append(idx)   # first valid saccade
                    break                 # only take the first one
                else:
                    break                 # skip to next stim

        idx_use = np.array(idx_buf, dtype=int)
        if idx_use.size == 0:
            continue

        ang = np.arctan2(dy[idx_use],
                            dx[idx_use])
        n_cond = len(idx_use)

        fig = plt.figure(figsize=(9, 5))
        gs  = gridspec.GridSpec(3, 2, width_ratios=[3, 2])
        ax_q = fig.add_subplot(gs[:, 0])
        ax_p = fig.add_subplot(gs[0, 1], polar=True)
        ax_l = fig.add_subplot(gs[1, 1])
        ax_t = fig.add_subplot(gs[2, 1]) if torsion_present else None

        ax_q.set_xlim(*X_LIM); ax_q.set_ylim(*Y_LIM)
        ax_q.set_xlabel('X (°)'); ax_q.set_ylabel('Y (°)')
        ax_q.set_title(f"{session_name}\n{eye_name} — {label} (n={n_cond})")

        cols = np.array([vector_to_rgb(a, max_abs) for a in ang])
        ax_q.quiver(eye_pos[idx_use, 0], eye_pos[idx_use, 1],
                    dx[idx_use], dy[idx_use],
                    angles='xy', scale_units='xy', scale=1,
                    color=cols, alpha=.5)

        plot_angle_distribution(ang, ax_p)
        plot_linear_histogram(ang, ax_l)

        if torsion_present:
            # Plot histogram of dtheta only for torsional saccades within window
            idx_buf_torsion = []

            # sort torsion saccade frames
            sorted_pairs_theta = sorted(zip(saccade_frames_theta, saccade_indices_theta))

            for f in frames:
                lower_bound = max(f + plot_window[0], 0)
                upper_bound = min(f + plot_window[-1], saccade_frames_theta.max())

                for sf, idx in sorted_pairs_theta:
                    if sf < lower_bound:
                        continue
                    elif sf <= upper_bound:
                        idx_buf_torsion.append(idx)
                        # break  # first torsional saccade
                    else:
                        break

            idx_torsion_use = np.array(idx_buf_torsion, dtype=int)
            if idx_torsion_use.size > 0:
                dtheta_torsion = dtheta[idx_torsion_use]
                ax_t.hist(dtheta_torsion, bins=20, color='purple', alpha=0.5, edgecolor='k')
                ax_t.set_title("Torsion angle distribution")
                ax_t.set_xlabel("deg/frame")
                ax_t.set_ylabel("Count")
                ax_t.set_xlim(-15, 15)

                # Add curved arrows for each torsional saccade
                for i in idx_torsion_use:
                    x0, y0 = eye_pos[i, 0], eye_pos[i, 1]
                    rotation_magnitude = np.abs(dtheta[i])
                    #print(f"Rotation magnitude for index {i}: {rotation_magnitude}")
                    curvature = -0.3 * np.sign(dtheta[i])  # direction of rotatio
                    arrow = FancyArrowPatch(
                        posA=(x0 - 0.7, y0-1),
                        posB=(x0 + 0.7, y0),
                        connectionstyle=f"arc3,rad={curvature}",
                        color='purple',
                        arrowstyle='->',
                        mutation_scale=10 + 2 * rotation_magnitude,  # scale by magnitude
                        linewidth=1.0,
                        alpha=0.8
                    )
                    ax_q.add_patch(arrow)

        fig.tight_layout()
        fname = f"{session_name}_{eye_name}_{label.replace('/','-')}.png"
        fig.savefig(results_dir / fname, dpi=300, bbox_inches='tight')


def plot_eye_fixations_between_cue_and_go_by_trial(
    eye_frame, eye_pos, eye_timestamp,
    cue_frame, cue_time, go_frame, go_time,
    max_interval_s=1.0,
    color_all='0.85', s_all=2, alpha_all=0.25,
    s_subset=5,  alpha_subset=0.9,
    cmap_name='tab20',
    results_dir=None, session_name=None, eye_name='Eye'
):
    # ---- coerce to arrays
    eye_ts     = np.asarray(eye_timestamp).ravel()
    eye_x      = np.asarray(eye_pos[:, 0]).ravel()
    eye_y      = np.asarray(eye_pos[:, 1]).ravel()
    cue_frame  = np.asarray(cue_frame).astype(int).ravel()
    cue_time   = np.asarray(cue_time).astype(float).ravel()
    go_frame   = np.asarray(go_frame).astype(int).ravel()
    go_time    = np.asarray(go_time).astype(float).ravel()

    # ---- sort cues & gos by time (carry frames alongside)
    ci = np.argsort(cue_time); cue_time, cue_frame = cue_time[ci], cue_frame[ci]
    gi = np.argsort(go_time);  go_time,  go_frame  = go_time[gi],  go_frame[gi]

    # ---- dedupe by time gaps (keeps first in each contiguous run)
    cue_time_on, cue_frame_on = cue_time, cue_frame
    go_time_on, go_frame_on = go_time, go_frame

    # ---- one-to-one time-based pairing: for each cue, take the NEXT go
    pairs_ct, pairs_gt, pairs_cf, pairs_gf, pairs_dt = [], [], [], [], []
    gptr = 0
    for ct, cf in zip(cue_time_on, cue_frame_on):
        while gptr < len(go_time_on) and go_time_on[gptr] < ct:
            gptr += 1
        if gptr >= len(go_time_on):
            break
        dt = float(go_time_on[gptr] - ct)
        pairs_ct.append(ct);  pairs_gt.append(go_time_on[gptr])
        pairs_cf.append(cf);  pairs_gf.append(int(go_frame_on[gptr]))
        pairs_dt.append(dt)
        gptr += 1  # consume this GO so it’s one-to-one

    pairs_ct = np.asarray(pairs_ct); pairs_gt = np.asarray(pairs_gt)
    pairs_cf = np.asarray(pairs_cf, dtype=int); pairs_gf = np.asarray(pairs_gf, dtype=int)
    pairs_dt = np.asarray(pairs_dt, dtype=float)

    # ---- filter by Δt window (seconds)
    valid_trials = (pairs_dt >= 0) & (pairs_dt < max_interval_s)

    # ---- plotting (use time to find eye samples; safer than frames)
    cmap = cm.get_cmap(cmap_name)
    base_colors = [cmap(i) for i in np.linspace(0, 1, 20)]
    color_cycle = cycle(base_colors)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(eye_x, eye_y, s=s_all, c=color_all, alpha=alpha_all, label='All eye centers')

    legend_handles = []
    trial_num = 0
    for ct, gt, ok, dt in zip(pairs_ct, pairs_gt, valid_trials, pairs_dt):
        if not ok:
            continue
        a = np.searchsorted(eye_ts, min(ct, gt), side='left')
        b = np.searchsorted(eye_ts, max(ct, gt), side='right')
        if b <= a:
            continue
        col = next(color_cycle)
        h = ax.scatter(eye_x[a:b], eye_y[a:b], s=s_subset, c=[col], alpha=alpha_subset,
                       label=f'Trial {trial_num} (Δt={dt:.2f}s)')
        legend_handles.append(h)
        trial_num += 1

    ax.set_aspect('equal')
    ax.set_xlabel('Eye center X (deg)')
    ax.set_ylabel('Eye center Y (deg)')
    ax.set_title(f'Eye centers: all vs. time-paired cue→go windows (<{max_interval_s:.1f}s)')

    # Trim legend clutter
    if len(legend_handles) > 10:
        ax.legend([ax.collections[0], *legend_handles[:10]],
                  ['All eye centers', *[lh.get_label() for lh in legend_handles[:10]]],
                  frameon=False, loc='best')
    else:
        ax.legend(frameon=False, loc='best')

    # optional save
    if results_dir is not None:
        results_dir = Path(results_dir); results_dir.mkdir(exist_ok=True, parents=True)
        fname = f"{session_name or 'session'}_{(eye_name or 'Eye').replace(' ', '')}_cue_go_timepaired.png"
        fig.savefig(results_dir / fname, dpi=300, bbox_inches='tight')

    # ---- diagnostics so you can sanity-check counts & thresholds
    print(f"Raw cues: {cue_time.size} → deduped: {cue_time_on.size}")
    print(f"Raw gos : {go_time.size}  → deduped: {go_time_on.size}")
    if pairs_dt.size:
        print(f"Paired trials: {pairs_dt.size} | dt min/median/max = "
              f"{np.nanmin(pairs_dt):.3f} / {np.nanmedian(pairs_dt):.3f} / {np.nanmax(pairs_dt):.3f} s")
        print(f"Passing (<{max_interval_s:.2f}s): {valid_trials.sum()} trials")

    return (pairs_cf, pairs_gf, pairs_ct, pairs_gt, pairs_dt, valid_trials,fig,ax)

def quantify_fixation_stability_vs_random(
    eye_timestamp, eye_pos,
    pairs_ct, pairs_gt, valid_trials,
    plot=True,
    rng_seed=0
):
    """
    Compare eye stability during fixation windows (cue->go for valid trials)
    to random, equal-duration windows drawn from the rest of the session.

    Returns a dict with arrays of per-window metrics and high-level means.
    Metrics per window:
      - mean_step_disp_px  : mean Euclidean step size (px)
      - mean_speed_px_s    : mean |velocity| (px/s)
      - net_drift_px       : |last - first| (px)
    """

    # --- coerce & sort eye samples by time
    ts = np.asarray(eye_timestamp, dtype=float).ravel()
    x  = np.asarray(eye_pos[:, 0]).ravel()
    y  = np.asarray(eye_pos[:, 1]).ravel()
    if not np.all(np.diff(ts) >= 0):
        order = np.argsort(ts)
        ts, x, y = ts[order], x[order], y[order]

    # --- build fixation windows from valid cue->go pairs
    ct = np.asarray(pairs_ct, dtype=float).ravel()
    gt = np.asarray(pairs_gt, dtype=float).ravel()
    ok = np.asarray(valid_trials, dtype=bool).ravel()

    fix_windows = [(c, g) for c, g, v in zip(ct, gt, ok) if v and (g > c)]
    if len(fix_windows) == 0:
        print("No valid fixation windows. Nothing to compute.")
        return None

    # Merge/clean fixation windows (ensure sorted, non-overlapping)
    fix_windows = sorted(fix_windows, key=lambda w: w[0])
    merged = []
    for s, e in fix_windows:
        if not merged or s > merged[-1][1]:
            merged.append([s, e])
        else:
            merged[-1][1] = max(merged[-1][1], e)  # merge overlaps
    fix_windows = [(s, e) for s, e in merged]

    # --- helper: compute metrics inside [t0, t1]
    def window_metrics(t0, t1):
        a = np.searchsorted(ts, t0, side='left')
        b = np.searchsorted(ts, t1, side='right')
        if b - a < 2:
            return np.nan, np.nan, np.nan
        dx = np.diff(x[a:b])
        dy = np.diff(y[a:b])
        dt = np.diff(ts[a:b])

        # valid finite steps with positive dt
        m = np.isfinite(dx) & np.isfinite(dy) & np.isfinite(dt) & (dt > 0)
        if not np.any(m):
            return np.nan, np.nan, np.nan

        step_disp = np.hypot(dx[m], dy[m])              # pixels
        speed     = step_disp / dt[m]                    # px/s
        drift     = np.hypot(x[b-1] - x[a], y[b-1] - y[a])  # pixels

        return float(step_disp.mean()), float(speed.mean()), float(drift)

    # --- compute fixation metrics per *original* (unmerged) window
    # (We’ll compare one random window per fixation with the same duration)
    orig_fix_windows = [(c, g) for c, g, v in zip(ct, gt, ok) if v and (g > c)]
    fix_len = np.array([g - c for c, g in orig_fix_windows], dtype=float)

    fix_mean_step = np.empty(len(orig_fix_windows))
    fix_mean_speed = np.empty(len(orig_fix_windows))
    fix_drift = np.empty(len(orig_fix_windows))
    for i, (c, g) in enumerate(orig_fix_windows):
        fix_mean_step[i], fix_mean_speed[i], fix_drift[i] = window_metrics(c, g)

    # --- build allowed (non-fixation) intervals across the whole session
    session_start, session_end = float(ts[0]), float(ts[-1])
    # complement of merged fixation windows
    allowed = []
    cursor = session_start
    for s, e in fix_windows:
        if s > cursor:
            allowed.append((cursor, s))
        cursor = max(cursor, e)
    if cursor < session_end:
        allowed.append((cursor, session_end))

    # convenience: function to draw a random start for a given duration
    rng = np.random.default_rng(rng_seed)
    def sample_random_window(duration):
        # find allowed intervals that can fit this duration
        candidates = [(a, b) for (a, b) in allowed if (b - a) >= duration]
        if not candidates:
            return None  # cannot fit (rare)
        a, b = candidates[rng.integers(0, len(candidates))]
        start = float(a) + rng.random() * float((b - a) - duration)
        return (start, start + duration)

    # --- draw one random window per fixation (equal duration) and compute metrics
    rnd_mean_step = np.empty(len(orig_fix_windows))
    rnd_mean_speed = np.empty(len(orig_fix_windows))
    rnd_drift = np.empty(len(orig_fix_windows))

    for i, L in enumerate(fix_len):
        rw = sample_random_window(L)
        if rw is None:
            rnd_mean_step[i] = rnd_mean_speed[i] = rnd_drift[i] = np.nan
        else:
            rnd_mean_step[i], rnd_mean_speed[i], rnd_drift[i] = window_metrics(*rw)

    # --- summarize (ignore NaNs)
    def nice_stats(arr):
        arr = np.asarray(arr, dtype=float)
        m = np.isfinite(arr)
        if not m.any():
            return np.nan, np.nan, 0
        vals = arr[m]
        return float(vals.mean()), float(vals.std(ddof=1) / np.sqrt(vals.size)), int(vals.size)

    ms_fix,  se_fix,  n_fix  = nice_stats(fix_mean_step)
    ms_rnd,  se_rnd,  n_rnd  = nice_stats(rnd_mean_step)
    sp_fix,  se_spf,  _      = nice_stats(fix_mean_speed)
    sp_rnd,  se_spr,  _      = nice_stats(rnd_mean_speed)
    dr_fix,  se_drf,  _      = nice_stats(fix_drift)
    dr_rnd,  se_drr,  _      = nice_stats(rnd_drift)

    print("=== Stability summary (mean ± s.e.m.) ===")
    print(f"Mean step displacement (px):  fix {ms_fix:.3f} ± {se_fix:.3f}   vs   rand {ms_rnd:.3f} ± {se_rnd:.3f}  (n={n_fix} pairs)")
    print(f"Mean speed (px/s):            fix {sp_fix:.3f} ± {se_spf:.3f}   vs   rand {sp_rnd:.3f} ± {se_spr:.3f}")
    print(f"Net drift (px):               fix {dr_fix:.3f} ± {se_drf:.3f}   vs   rand {dr_rnd:.3f} ± {se_drr:.3f}")

    # --- optional quick plot
    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
        pairs = [
            ("Mean step (deg)", fix_mean_step, rnd_mean_step),
            ("Mean speed (deg/s)", fix_mean_speed, rnd_mean_speed),
            ("Net drift (deg)", fix_drift, rnd_drift),
        ]
        for ax, (title, a, b) in zip(axes, pairs):
            m = np.isfinite(a) & np.isfinite(b)
            ax.scatter(a[m], b[m], s=10, alpha=0.6)
            # y=x reference line
            lo = np.nanmin(np.concatenate([a[m], b[m]]))
            hi = np.nanmax(np.concatenate([a[m], b[m]]))
            if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
                ax.plot([lo, hi], [lo, hi], linestyle='--', linewidth=1, alpha=0.5)
                ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
            ax.set_xlabel("Fixation"); ax.set_ylabel("Random")
            ax.set_title(title)
            ax.set_aspect('equal', adjustable='box')
        fig.suptitle("Fixation vs. random windows (paired, equal duration)")
    else:
        fig = None

    return {
        "fix_mean_step_px": fix_mean_step,
        "rnd_mean_step_px": rnd_mean_step,
        "fix_mean_speed_px_s": fix_mean_speed,
        "rnd_mean_speed_px_s": rnd_mean_speed,
        "fix_net_drift_px": fix_drift,
        "rnd_net_drift_px": rnd_drift,
        "summary": {
            "mean_step_fix_mean±sem": (ms_fix, se_fix, n_fix),
            "mean_step_rand_mean±sem": (ms_rnd, se_rnd, n_rnd),
            "mean_speed_fix_mean±sem": (sp_fix, se_spf),
            "mean_speed_rand_mean±sem": (sp_rnd, se_spr),
            "net_drift_fix_mean±sem": (dr_fix, se_drf),
            "net_drift_rand_mean±sem": (dr_rnd, se_drr),
        },
        "figure": fig,
    }





## Extract relevant files from filepath

In [ ]:
# determine which camera is used based on the folder name and cut off date
camera_side = determine_camera_side(folder_path)
eye_name = f"{camera_side} Eye"  # e.g. "Left camera" or "Right camera"
print(f"Using camera side: {camera_side}")



print(f"Scanning folder: {folder_path}")
print(f"Found {len(os.listdir(folder_path))} files")

# Scan the folder for specific files
for f in os.listdir(folder_path):
    f_lower = f.lower()
    full_path = os.path.join(folder_path, f)

    if 'imu' in f_lower:
        IMU_file = full_path
    if 'camera' in f_lower:
        camera_file = full_path
    if 'go' in f_lower:
        go_file = full_path
    if f"ellipse_center_XY_{camera_side}".lower() in f_lower:
        ellipse_center_XY_file = full_path
    if f"origin_of_eyecoordinate_{camera_side}".lower() in f_lower:
        origin_of_eye_coordinate_file = full_path
    if f"vdaxis_{camera_side}".lower() in f_lower:
        vdaxis_file = full_path
    if f"torsion_{camera_side}".lower() in f_lower:
        torsion_file = full_path
    if 'endoftrial' in f_lower:
        end_of_trial_file = full_path
    if 'cue' in f_lower:
        cue_file = full_path




## Extract stim and eye position data 

In [ ]:
## Read the camera data and map between camera TTL (for saccades) and Bonsai TTLs (for frames)
camera_data = np.genfromtxt(camera_file, delimiter=',', skip_header=1, dtype=np.float64)
[bonsai_frame, bonsai_time] = camera_data[:, 0], camera_data[:, 1]
bonsai_frame = bonsai_frame.astype(int)  # Convert bonsai_frame to integer type


### Read the go file for the start of stim in the trial 
new_go_data_format=0
go_data = np.genfromtxt(clean_csv(go_file), delimiter=',', skip_header=1, dtype=np.float64)
if go_data.shape[1]>3:
    new_go_data_format = 1
    [go_frame, go_time, go_direction_x,go_direction_y] = go_data[:, 0], go_data[:, 1], go_data[:, 2], go_data[:,3]
else:
    [go_frame, go_time, go_direction] = go_data[:, 0], go_data[:, 1], go_data[:, 2]
go_frame = go_frame.astype(int)  # Convert go_frame to integer type

### Read the ellipse center XY file  
ellipse_center_XY_data = np.genfromtxt(clean_csv(ellipse_center_XY_file), delimiter=',', skip_header=1, dtype=np.float64)
[eye_frame,eye_timestamp,eye_x,eye_y] = ellipse_center_XY_data[:, 0], ellipse_center_XY_data[:, 1], ellipse_center_XY_data[:, 2], ellipse_center_XY_data[:, 3]
eye_frame = eye_frame.astype(int)  # Convert eye_frame to integer type
eye_x = interpolate_nans(eye_x)  # Interpolate NaN values in eye_x
eye_y = -1*interpolate_nans(eye_y)  # Interpolate NaN values in eye_y

### Read the origin of eye coordinate file
origin_of_eye_coordinate_data = np.genfromtxt(clean_csv(origin_of_eye_coordinate_file), delimiter=',', skip_header=1, dtype=np.float64)
[origin_frame,o_ts,l_x,l_y,r_x,r_y] = origin_of_eye_coordinate_data[:, 0], origin_of_eye_coordinate_data[:, 1], origin_of_eye_coordinate_data[:, 2], origin_of_eye_coordinate_data[:, 3], origin_of_eye_coordinate_data[:, 4], origin_of_eye_coordinate_data[:, 5]
origin_frame = origin_frame.astype(int)  # Convert origin_frame_r to integer type
l_x = interpolate_nans(l_x)  # Interpolate NaN values in l_rx
r_x = interpolate_nans(r_x)  # Interpolate NaN values in r_rx 
l_y = interpolate_nans(l_y)  # Interpolate NaN values in l_ry
r_y = interpolate_nans(r_y)  # Interpolate NaN values in r_ry


## Read the torsion data - this is used for torsion detection
torsion_data = np.genfromtxt(clean_csv(torsion_file), delimiter=',', skip_header=1, dtype=np.float64)
[torsion_frame, torsion_ts, torsion] = torsion_data[:, 0], torsion_data[:, 1], torsion_data[:, 2]
torsion_frame = torsion_frame.astype(int)   # Convert torsion_frame to integer type
# Interpolate NaN values        
torsion = interpolate_nans(torsion)


## Read the vertical (VD) axis data - this is used for blink detection
vdaxis_data = np.genfromtxt(clean_csv(vdaxis_file),delimiter=',',skip_header=1,dtype=np.float64)
[vd_frame,vd_ts,vd_lx,vd_ly,vd_rx,vd_ry] = vdaxis_data[:,0],vdaxis_data[:,1],vdaxis_data[:,2],vdaxis_data[:,3],vdaxis_data[:,4],vdaxis_data[:,5]
vd_frame = vd_frame.astype(int)
# Interpolate NaN values
vd_lx = interpolate_nans(vd_lx)
vd_ly = interpolate_nans(vd_ly)
vd_rx = interpolate_nans(vd_rx)
vd_ry = interpolate_nans(vd_ry)

        
### Read the IMU data for the accelerometer and gyroscope
imu_data = np.genfromtxt(IMU_file, delimiter=',', skip_header=1, dtype=np.float64)
[imu_time,a_x,a_y,a_z,g_x,g_y,g_z,m_x,m_y,m_z] = imu_data[:, 0], imu_data[:, 1], imu_data[:, 2], imu_data[:, 3], imu_data[:, 4], imu_data[:, 5], imu_data[:, 6], imu_data[:, 7], imu_data[:, 8], imu_data[:, 9]
imu_time = imu_time.astype(np.float64)  # Ensure imu_time is in float64 format
# Interpolate NaN values in IMU data
a_x = interpolate_nans(a_x)
a_y = interpolate_nans(a_y)
a_z = interpolate_nans(a_z)
g_x = interpolate_nans(g_x)
g_y = interpolate_nans(g_y)
g_z = interpolate_nans(g_z)
m_x = interpolate_nans(m_x)
m_y = interpolate_nans(m_y)
m_z = interpolate_nans(m_z)


### Read the endoftrial file- This file tells us when the trial ends, stim direction, eye movement direction, torsion angle, and whether the trial was successful
try:
    end_of_trial_data = np.genfromtxt(clean_csv(end_of_trial_file), delimiter=',', skip_header=1, dtype=np.float64)
    [end_of_trial_frame, end_of_trial_ts, trial_stim_direction, trial_eye_movement_direction, trial_torsion_angle, trial_success] = end_of_trial_data[:, 0], end_of_trial_data[:, 1], end_of_trial_data[:, 2], end_of_trial_data[:, 3], end_of_trial_data[:, 4], end_of_trial_data[:, 5]
    end_of_trial_frame = end_of_trial_frame.astype(int)  # Convert end_of_trial_frame to integer type
    # Interpolate NaN values in trial_torsion_angle
    trial_torsion_angle = interpolate_nans(trial_torsion_angle)
    trial_eye_movement_direction = interpolate_nans(trial_eye_movement_direction)
except IndexError:
    end_of_trial_data = np.genfromtxt(clean_csv(end_of_trial_file), delimiter=',', skip_header=1, dtype=np.float64)
    [end_of_trial_frame, end_of_trial_ts, trial_stim_direction, trial_eye_movement_direction, trial_success] = end_of_trial_data[:, 0], end_of_trial_data[:, 1], end_of_trial_data[:, 2], end_of_trial_data[:, 3], end_of_trial_data[:, 4]
    end_of_trial_frame = end_of_trial_frame.astype(int)  # Convert end_of_trial_frame to integer type

    trial_eye_movement_direction = interpolate_nans(trial_eye_movement_direction)
except ValueError:
    print("No end of trial data found. Skipping this step.")
for t in range(len(trial_success)):
    if trial_success[t] == 0:
        if trial_eye_movement_direction[t] != -1:
            trial_success[t] = -1   # Incorrect trial

# --- Read the cue file (every-frame logging) and keep only trial onsets ---
cue_data = np.genfromtxt(clean_csv(cue_file), delimiter=',', skip_header=1, dtype=np.float64)

cue_frame_raw     = cue_data[:, 0].astype(int)
cue_time_raw      = cue_data[:, 1].astype(float)
cue_direction_raw = cue_data[:, 2]  # keep dtype as-is (often int/float)

# Sort by time to be safe (carry frames/directions along)
order = np.argsort(cue_time_raw)
cue_time_raw      = cue_time_raw[order]
cue_frame_raw     = cue_frame_raw[order]
cue_direction_raw = cue_direction_raw[order]

# Define what counts as a 'new trial' gap between consecutive cue rows
TRIAL_GAP_S = 1.5  # <-- adjust to 2–3 if your inter-trial gap is longer

# Keep only the FIRST row after each large time jump (trial onset)
onset_idx = np.r_[0, np.where(np.diff(cue_time_raw) > TRIAL_GAP_S)[0] + 1]
cue_frame     = cue_frame_raw[onset_idx]
cue_time      = cue_time_raw[onset_idx]
cue_direction = cue_direction_raw[onset_idx]

print(f"Detected {cue_frame.size} cue onsets from {cue_frame_raw.size} cue rows (gap > {TRIAL_GAP_S}s).")

# --- Align lengths with GO events (1 line per trial) ---
if len(cue_frame) != len(go_frame):
    n = min(len(cue_frame), len(go_frame))
    if len(cue_frame) > len(go_frame):
        print(f"Warning: {len(cue_frame)} cue onsets but {len(go_frame)} GO rows; truncating cues to {n}.")
        cue_frame, cue_time, cue_direction = cue_frame[:n], cue_time[:n], cue_direction[:n]
    else:
        print(f"Warning: {len(cue_frame)} cue onsets but {len(go_frame)} GO rows; truncating GO to {n}.")
        go_frame, go_time = go_frame[:n], go_time[:n]

## Sanity check: How far apart are the stimuli?  


In [ ]:

d_frames = np.diff(go_frame)    # successive differences (frames)
d_sec = d_frames / ttl_freq

fig =plt.figure(figsize=(8,3))
plt.plot(d_sec, marker='o')
plt.xlabel('Stimulus index')
plt.ylabel('Δtime (s) to next stim')
plt.title('Seconds between successive Go Stims')
plt.grid(alpha=.3)
plt.tight_layout()
plt.show()

# optional: save alongside other figures
prob_fname = f"{session_name}_{eye_name}_Stim_Interval.png"
fig.savefig(results_dir / prob_fname, dpi=300, bbox_inches='tight')

## Analyze and plot saccade statistics

In [ ]:

# --------------------------------------------------------------
# ONE analyse-&-plot call for the Eye saccades
saccades = detect_saccades(
    l_x, l_y, r_x, r_y,
    eye_x, eye_y,
    eye_frame,
    cal,
    blink_detection = blink_detection,
    vd_axis_lx = vd_lx, vd_axis_ly = vd_ly,
    vd_axis_rx =vd_rx, vd_axis_ry = vd_ry,
    saccade_threshold       = saccade_thresh,
    blink_velocity_threshold= blink_thresh,
    torsion_angle = torsion,
    saccade_threshold_torsion = torsion_velocity_thresh,
)

saccades["stim_frames"], stim_type = organize_stims(
    go_frame,
    go_dir_x = go_direction_x,
    go_dir_y = go_direction_y,
)

sort_plot_saccades(
    saccades,
    saccade_window= saccade_win*ttl_freq,
    session_path = folder_path,
    stim_type    = stim_type,
    eye_name     = eye_name,

)


## Look at probability of saccades as a function of stimulus onset

In [ ]:
go_frame        = go_frame = np.array(go_frame).flatten()
saccade_frames  = saccades["saccade_frames_xy"]
ttl_freq        = 60
t_window_s      = 2
bin_ms          = 50

total_duration_frames = None

# ─── Setup ─────────────────────────────────────────────
frame_win   = int(t_window_s * ttl_freq)
bin_frames  = int((bin_ms / 1000) * ttl_freq)
bins        = np.arange(-frame_win, frame_win + 1, bin_frames)
t_sec       = (bins[:-1] + bin_frames / 2) / ttl_freq

# ─── Estimate baseline saccade probability ─────────────
if total_duration_frames is None:
    total_duration_frames = saccade_frames.max()

total_bins = total_duration_frames / bin_frames
baseline_rate = len(saccade_frames) / total_bins  # saccades per bin

# ─── Helper: peri-stimulus histogram ──────────────────
def peristim_change(stim_frames):
    rel_times = []
    for f0 in stim_frames:
        nearby = saccade_frames[
            (saccade_frames >= f0 - frame_win) &
            (saccade_frames <= f0 + frame_win)
        ]
        rel_times.extend(nearby - f0)
    rel_times = np.array(rel_times)

    counts, _ = np.histogram(rel_times, bins=bins)
    prob_bin = counts / len(stim_frames)  # saccades per bin per stim
    change = 100 * (prob_bin - baseline_rate) / baseline_rate
    return change

# ─── Group by direction ────────────────────────────────
go_dir_x = np.nan_to_num(go_direction_x)
go_dir_y = np.nan_to_num(go_direction_y)

stim_all = go_frame
stim_L   = go_frame[go_dir_x < 0]
stim_R   = go_frame[go_dir_x > 0]
stim_D   = go_frame[go_dir_y < 0]
stim_U   = go_frame[go_dir_y > 0]

# ─── Compute changes ───────────────────────────────────
ch_all = peristim_change(stim_all)
ch_L   = peristim_change(stim_L)
ch_R   = peristim_change(stim_R)
ch_D   = peristim_change(stim_D)
ch_U   = peristim_change(stim_U)

# ─── Plot ──────────────────────────────────────────────
# Plotting
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 4), sharex=True)

# Top: all stimuli
ax1.plot(t_sec, ch_all, label="All stimuli", color="black", lw=2)
ax1.axvline(0, color='k', ls='--', alpha=.5)
ax1.axhline(0, color='gray', ls=':', lw=1)
ax1.set_title('Stim-locked saccade probability (relative to baseline saccade rate)')
ax1.grid(alpha=.3)
ax1.set_ylim(-100,200)
ax1.legend()

# Bottom: each direction
if ch_L is not None:
    ax2.plot(t_sec, ch_L, label=f"Left (n={len(stim_L)})", color='green')
if ch_R is not None:
    ax2.plot(t_sec, ch_R, label=f"Right (n={len(stim_R)})", color='pink')
if ch_D is not None:
    ax2.plot(t_sec, ch_D, label=f"Down (n={len(stim_D)})", color='blue')
if ch_U is not None:
    ax2.plot(t_sec, ch_U, label=f"Up (n={len(stim_U)})", color='red')

ax2.axvline(0, color='k', ls='--', alpha=.5)
ax2.axhline(0, color='gray', ls=':', lw=1)
ax2.set_xlabel('Time from stimulus onset (s)')
ax2.set_title('Modulation by direction')
ax2.set_ylim(-100,200)
ax2.grid(alpha=.3)
#ax2.legend()

# Shared y-axis label
fig.text(0.01, 0.5, '% Change ',
         va='center', rotation='vertical', fontsize=11)

fig.tight_layout()


# optional: save alongside other figures
prob_fname = f"{session_name}_{eye_name}_StimLockedProb.png"
fig.savefig(results_dir / prob_fname, dpi=300, bbox_inches='tight')

## Sanity check: Plot an overlay of Go-stims and Saccades 

In [ ]:
# ============================================================
# Timeline plot: saccades (grey) + colour-coded stimuli
# ============================================================
# -----------------------------------------------------------------
# Assumes these objects already exist in the workspace
# -----------------------------------------------------------------
# right["saccade_frames"]   – Bonsai frame IDs of all saccades
# go_frame                  – Bonsai frame IDs of stimuli
# go_dir_x, go_dir_y        – direction codes (can be None)
# ttl_freq                  – camera TTL rate (Hz)
# results_dir               – Path(folder_path) / "Results"
# -----------------------------------------------------------------

# 1) prep data ----------------------------------------------------
saccade_frames = np.asarray(saccades["saccade_frames_xy"], dtype=int)

# fallback arrays if dir arrays are missing
gx = go_dir_x if (go_dir_x is not None) else np.zeros_like(go_frame)
gy = go_dir_y if (go_dir_y is not None) else np.zeros_like(go_frame)


# ── 1-bis. count how many of each stimulus -----------------------------
eps = 1e-6                      # tolerance for “zero”
is_left   = (np.abs(gy) < eps) & (gx < -eps)
is_right  = (np.abs(gy) < eps) & (gx >  eps)
is_down   = (np.abs(gx) < eps) & (gy < -eps)
is_up     = (np.abs(gx) < eps) & (gy >  eps)

n_left, n_right = is_left.sum(),  is_right.sum()
n_down, n_up    = is_down.sum(),  is_up.sum()


# palette mapping
palette = {'L': 'green', 'R': 'pink', 'D': 'blue', 'U': 'red', 'NA': 'gray'}

# build colour list per stimulus
colors = []
for x, y in zip(gx, gy):
    if abs(y) > 1e-6:                       # Up / Down has priority
        colors.append(palette['U' if y > 0 else 'D'])
    elif abs(x) > 1e-6:                     # Left / Right
        colors.append(palette['R' if x > 0 else 'L'])
    else:
        colors.append(palette['NA'])

# sort frames & colours together
order      = np.argsort(go_frame)
t_stim     = go_frame[order] / ttl_freq
colors     = [colors[i] for i in order]

# convert saccade frames to seconds
t_sacc = np.sort(saccade_frames) / ttl_freq

# 2) plot ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 2.5))

# saccades: grey vertical ticks at y = 0
ax.vlines(t_sacc, -0.1, 0.1, colors='0.25', linewidth=1)

# stimuli: colour ticks at y = 1
ax.vlines(t_stim, 0.9, 1.1, colors=colors, linewidth=2)

# axes formatting
ax.set_yticks([0, 1])
ax.set_yticklabels(['Saccade', 'Stim'])
ax.set_xlabel('Time (s)')
ax.set_title('Timeline of saccades and stimuli')
ax.set_xlim(t_sacc.min() - 1, t_sacc.max() + 1)
ax.set_ylim(-0.5, 1.5)
ax.grid(axis='x', alpha=.3)

# legend
handles = [
    mlines.Line2D([], [], color='0.25', marker='|', ls='', markersize=10,
                  label='Saccade'),
    mlines.Line2D([], [], color='green', marker='|', ls='', markersize=10,
                  label=f'Stim Left  (n={n_left})'),
    mlines.Line2D([], [], color='pink',  marker='|', ls='', markersize=10,
                  label=f'Stim Right (n={n_right})'),
    mlines.Line2D([], [], color='blue',  marker='|', ls='', markersize=10,
                  label=f'Stim Down  (n={n_down})'),
    mlines.Line2D([], [], color='red',   marker='|', ls='', markersize=10,
                  label=f'Stim Up    (n={n_up})')
]
ax.legend(handles=handles, loc='upper right', ncol=5, fontsize=9, framealpha=.9)

plt.tight_layout()


# optional: save alongside other figures
prob_fname = f"{session_name}_{eye_name}_timeline_saccade_vs_stim.png"
fig.savefig(results_dir / prob_fname, dpi=300, bbox_inches='tight')


## Plot how many stims produced at least one saccade

In [ ]:
# ============================================================
#  Probability of ≥1 saccade within 0.5 s of each stimulus


# ----- parameters -------------------------------------------
win     = saccade_win                  # seconds after onset
w_frames = int(win * ttl_freq)  # convert to frames

# direction masks
gx = go_direction_x if go_direction_x is not None else np.zeros_like(go_frame)
gy = go_direction_y if go_direction_y is not None else np.zeros_like(go_frame)

dir_info = {
    'Left' :  (gx < -1e-6,  'green'),
    'Right':  (gx >  1e-6,  'pink'),
    'Down' :  (gy < -1e-6,  'blue'),
    'Up'   :  (gy >  1e-6,  'red')
}

labels, probs, colors = [], [], []

for label, (mask, col) in dir_info.items():
    stim_frames = go_frame[mask]
    n_stim      = len(stim_frames)
    if n_stim == 0:
        continue                                 # skip if this direction absent
    # check each stimulus: does ANY saccade happen within +win seconds?
    has_sacc = [( (saccade_frames >= f) & (saccade_frames <= f + w_frames) ).any()
                for f in stim_frames]
    prob = np.mean(has_sacc)                    # fraction of stimuli with ≥1 sac
    labels.append(label)
    probs.append(prob)
    colors.append(col)
    print(f"{label:5s}: {prob*100:5.1f}%  ({sum(has_sacc)}/{n_stim} stimuli)")

# ----- bar chart --------------------------------------------
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(labels, probs, color=colors, edgecolor='k')
ax.set_ylim(0, 1)
ax.set_ylabel(f"P(saccade within {win}s)")
ax.set_title(f"Probability of a saccade in first {win} s after stimulus")
ax.grid(axis='y', alpha=.3)
plt.tight_layout()

# ----- save (optional) --------------------------------------


# optional: save alongside other figures
prob_fname = f"{session_name}_{eye_name}_saccade_prob_within_{win*1000:.0f}ms.png"
fig.savefig(results_dir / prob_fname, dpi=300, bbox_inches='tight')



## Fixation: Plot the fixation eye positions and the distribution.

Get the fixation position from the eye positions just before the stimulus onset (Go Cue)


In [ ]:
## For each go frame, look at the last available frame before the go frame in the eye position traces.
eye_position_during_fixation=[]
eye_position_during_fixation_success = []
## Basic sanity check regarding trials statistics collected from different sources
if (len(end_of_trial_frame) != len(go_frame)):
    print(f"Warning: Number of end of trial frames ({len(end_of_trial_frame)}) does not match number of go frames ({len(go_frame)}).")

for i, f in enumerate(go_frame[:len(trial_success)]):
    #last_frame_before_go = eye_frame[eye_frame < f][-31:-1]  # last 30 eye frame before stim
    #eye_pos = np.mean(saccades["eye_pos"][np.where(eye_frame == last_frame_before_go)[0]], axis=0)  # average eye position at that frame
    eye_pos = np.mean(saccades["eye_pos"][np.where(eye_frame <f)[0][-7:-1]], axis=0)  # average eye position at the last 30 frames before the go frame
    if len(eye_pos) == 0:
        print(f"Warning: No eye position data found for go frame {f}. Skipping this frame.")
        continue
    eye_position_during_fixation.append(eye_pos)
    if trial_success[i] == 1:
        eye_position_during_fixation_success.append(eye_pos)
eye_position_during_fixation = np.array(eye_position_during_fixation)
eye_position_during_fixation_success = np.array(eye_position_during_fixation_success)
# Ratio of the total spread of the eye positions during fixation to the total spread of all eye positions
eye_pos_all = saccades["eye_pos"]  
spread_fixation = np.std(eye_position_during_fixation, axis=0)
spread_all = np.std(eye_pos_all, axis=0)
ratio_spread = spread_fixation / spread_all
print(f"Ratio of spread during fixation to all eye positions: {ratio_spread}")


# Plot all the eye positions during the session first and then the eye positions during fixation
fig = plt.figure(figsize=(8, 6))
plt.scatter(saccades["eye_pos"][:, 0], saccades["eye_pos"][:, 1], color='red', alpha=0.1, label='All Eye Positions')
plt.scatter(eye_position_during_fixation[:, 0], eye_position_during_fixation[:, 1], color='blue', alpha=0.4, label='Eye Positions During Fixation')
plt.scatter(eye_position_during_fixation_success[:, 0], eye_position_during_fixation_success[:, 1], color='green', alpha=0.5, label='Eye Positions During Fixation (Successful Trials)')
plt.xlabel('X Position (deg)')
plt.ylabel('Y Position (deg)')
plt.title('Eye Positions in the orbit during the whole session and during fixation')
plt.legend()
plt.grid()
plt.show()



## Fixation: Compare eye movements during fixation period to randomly selected, matched duration, other times

In [ ]:
rng = np.random.default_rng(123)  # set a seed for reproducibility
cue_frame_jit = (cue_frame + rng.integers(0, 101, size=cue_frame.shape)).astype(int)
cue_time_jit  = cue_time + rng.uniform(0.0, 5.0, size=cue_time.shape)

(pairs_cf, pairs_gf, pairs_ct, pairs_gt, pairs_dt, valid_trials,fig,ax)= plot_eye_fixations_between_cue_and_go_by_trial(
    eye_frame=eye_frame, eye_pos=saccades["eye_pos"], eye_timestamp=eye_timestamp,
    cue_frame=cue_frame, cue_time=cue_time,
    #cue_frame=cue_frame_jit, cue_time=cue_time_jit,    
    go_frame=go_frame,  go_time=go_time,
    max_interval_s=1,
    results_dir=results_dir, session_name=session_name, eye_name=eye_name
)

# Now quantify stability vs random (and show a small paired scatter summary)
stats = quantify_fixation_stability_vs_random(
    eye_timestamp=eye_timestamp,
    eye_pos=saccades["eye_pos"],
    pairs_ct=pairs_ct, pairs_gt=pairs_gt,
    valid_trials=valid_trials,
    plot=True,     # set False if you only want numbers
    rng_seed=0
)

In [ ]:
## This section is only for the fixation experiments. Might throw an error if the data is not from a fixation experiment.
fixation_experiment = False
#----------------------------------------------------------------------------------------------------------
## Divide the successful fixation trials based on how long it took the animal to reach fixation point. We calculate this from the difference between the cue frame and go_frame.
if fixation_experiment:
    fixation_time_theshold = 0.75  # seconds
    total_fixation_time_per_trial = go_time - cue_time
    #plt.hist(total_fixation_time_per_trial, bins=20, color='gray', alpha=0.7)
    mask_short_fixation = total_fixation_time_per_trial <= fixation_time_theshold
    mask_long_fixation = total_fixation_time_per_trial > fixation_time_theshold
    short_fixation_frames = []
    for i, f in enumerate(cue_frame[mask_short_fixation]):
        short_fixation_frames.append(eye_frame[np.where(eye_frame>=f)[0][0]:np.where(eye_frame<=go_frame[mask_short_fixation][i])[0][-1]])
    short_fixation_frames = np.array(short_fixation_frames, dtype=object)  # array of arrays
    short_fixation_eye_positions = []
    for frames in short_fixation_frames:
        positions = []
        for fr in frames:
            pos = saccades["eye_pos"][np.where(eye_frame == fr)[0]]
            if len(pos) > 0:
                positions.append(pos[0])
        positions = np.array(positions)
    # positions = positions - np.mean(positions, axis=0)  # center the positions around the mean
        short_fixation_eye_positions.append(positions)
    short_fixation_eye_positions = np.array(short_fixation_eye_positions, dtype=object)  # array of arrays
    long_fixation_frames = []
    for i, f in enumerate(cue_frame[mask_long_fixation]):
        long_fixation_frames.append(eye_frame[np.where(eye_frame>=f)[0][0]:np.where(eye_frame<=go_frame[mask_long_fixation][i])[0][-1]])
    long_fixation_frames = np.array(long_fixation_frames, dtype=object)  # array of arrays
    long_fixation_eye_positions = []    
    for frames in long_fixation_frames:
        positions = []
        for fr in frames:
            pos = saccades["eye_pos"][np.where(eye_frame == fr)[0]] 
            if len(pos) > 0:
                positions.append(pos[0])
        positions = np.array(positions)
    # positions = positions - np.mean(positions, axis=0)  # center the positions around the mean
        long_fixation_eye_positions.append(np.array(positions))
    long_fixation_eye_positions = np.array(long_fixation_eye_positions, dtype=object)  # array of arrays
    # Plot the eye positions during short and long fixation trials
    #fig = plt.figure(figsize=(12, 5))
    short_fixation_all_positions = np.vstack(short_fixation_eye_positions)
    long_fixation_all_positions = np.vstack(long_fixation_eye_positions)

    ## Let's look at some randomly selected windows from the whole sessions for control
    control_window_eye_positions = []
    control_window_size = int(fixation_time_theshold * ttl_freq)  # in frames
    control_window_start = np.random.choice



    #----------------------------------------------------------------------------------------------------------
    ## This commented section is for side-by-side plots
    #----------------------------------------------------------------------------------------------------------
    # fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    # ax1.scatter(short_fixation_all_positions[:, 0], short_fixation_all_positions[:, 1], color='blue', alpha=0.3)
    # ax1.set_title(f'Eye Positions During Short Fixation Trials (<= {fixation_time_theshold}s)')
    # ax1.set_xlabel('X Position (deg)')
    # ax1.set_ylabel('Y Position (deg)')  
    # ax1.grid()
    # ax2.scatter(long_fixation_all_positions[:, 0], long_fixation_all_positions[:, 1], color='orange', alpha=0.3)
    # ax2.set_title(f'Eye Positions During Long Fixation Trials (> {fixation_time_theshold}s)')
    # ax2.set_xlabel('X Position (deg)')
    # ax2.set_ylabel('Y Position (deg)')  
    # ax2.grid()
    # plt.show()
    #----------------------------------------------------------------------------------------------------------
    # This section is for overlayed plots
    #----------------------------------------------------------------------------------------------------------
    fig = plt.figure(figsize=(8, 6))
    plt.scatter(saccades["eye_pos"][:, 0], saccades["eye_pos"][:, 1], color='red', alpha=0.1, label='All Eye Positions')
    plt.scatter(long_fixation_all_positions[:, 0], long_fixation_all_positions[:, 1], color='orange', alpha=0.3, label=f'Long Fixation (> {fixation_time_theshold}s)')
    plt.scatter(short_fixation_all_positions[:, 0], short_fixation_all_positions[:, 1], color='blue', alpha=0.3, label=f'Short Fixation (<= {fixation_time_theshold}s)')
    plt.xlabel('X Position (deg)')
    plt.ylabel('Y Position (deg)')
    plt.title('Eye Positions During Short and Long Fixation Trials')
    plt.legend()
    plt.grid()
    plt.show()



## Performance: Plot the perforance of the animal throughout the session.

 Useful to see when the animal is getting distracted or tired. Also useful to see if the animal is learning the task.

In [ ]:
## Plot the moving average of the trial success rate over a sliding window of 20 trials
window_size = 10
trial_success = np.array(trial_success, dtype=int)  # Ensure it's an integer array
moving_avg_success = np.convolve(trial_success, np.ones(window_size)/window_size, mode='valid')
fig = plt.figure(figsize=(10, 5))
plt.plot(moving_avg_success, color='blue', label='Moving Average Success Rate')
plt.xlabel('Trial Index')
plt.ylabel('Success Rate (Moving Average)')
plt.title(f'Moving Average of Trial Success Rate (Window Size: {window_size})')
plt.axhline(y=0.5, color='red', linestyle='--', label='50% Success Rate')
plt.legend()
plt.grid()
plt.tight_layout()

## Anti-saccade: Plot the performance of the animal in the antisaccade task.

 Correct, Incorrect, Missed trials. Magnitude and timing of saccades in correct and incorrect trials. Percentage of correct trials that had first saccade in the incorrect direction.

In [ ]:
## First plot the percentage of the correct, incorrect, and missed trials
# end_of_trial_frame, end_of_trial_ts, trial_stim_direction, trial_eye_movement_direction, trial_torsion_angle, trial_success
num_trials = len(end_of_trial_frame)
num_correct = np.sum(trial_success == 1)
num_missed = np.sum(trial_success == 0)
num_incorrect = np.sum(trial_success == -1)
fig, ax = plt.subplots(figsize=(8, 6))
labels = ['Correct Trials', 'Missed Trials', 'Incorrect Trials']
sizes = [num_correct, num_missed, num_incorrect]
# Normalize sizes to percentages
# Percentages to be written on the bar chart
percentages = [f"{size/num_trials*100:.1f}%" for size in sizes]
colors = ['green', 'orange', 'red']
bars = ax.bar(labels, sizes, color=colors)
# Add percentage labels on top of the bars
for bar, percentage in zip(bars, percentages):
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 0.5, percentage, ha='center', va='bottom')
ax.set_ylabel('Number of Trials')
ax.set_title('Trial Outcomes')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
# Save the figure
trial_outcome_fname = f"{session_name}_{eye_name}_Trial_Outcomes.png"   
fig.savefig(results_dir / trial_outcome_fname, dpi=300, bbox_inches='tight')


In [ ]:
## Plot the comparison of saccade magnitude and direction between correct, incorrect, and missed trials
saccade_speeds_correct_all = []
saccade_speeds_correct_latency_all = []
saccade_speeds_incorrect_before_correct_all = []
saccade_latency_incorrect_before_correct_all = []
saccade_speeds_incorrect_all = []
saccade_speeds_incorrect_latency_all = []
saccade_speeds_correct_first_saccade_all = []
saccade_latency_correct_first_saccade_all = []
for i, f in enumerate(go_frame[:len(end_of_trial_frame)]):  # Loop through each go frame
    if trial_success[i] == 1: ## Correct trial
        # Find the number of saccade between go frame and end of the trial frame
        saccade_indices = np.where((saccades['saccade_frames_xy'] >=f) & (saccades['saccade_frames_xy'] <= end_of_trial_frame[i]))[0]
        if len(saccade_indices) > 0:
            saccade_speeds_correct = np.linalg.norm(saccades['eye_vel'][saccades['saccade_indices_xy'][saccade_indices]], axis=1)
            saccade_directions_correct = np.rad2deg(np.arctan2(saccades['eye_vel'][saccades['saccade_indices_xy'][saccade_indices], 1], saccades['eye_vel'][saccades['saccade_indices_xy'][saccade_indices], 0]))
            saccade_speeds_correct_all.append(saccade_speeds_correct[-1])
            saccade_speeds_correct_latency_all.append((saccades['saccade_frames_xy'][saccade_indices[-1]] - f)/60.0)  # latency of the last saccade in the trial
            if len(saccade_indices) > 1:
                saccade_speeds_incorrect_before_correct_all.append(saccade_speeds_correct[0])  # first incorrect saccade before the correct one
                saccade_latency_incorrect_before_correct_all.append((saccades['saccade_frames_xy'][saccade_indices[0]] - f)/60.0)  # latency of the first saccade in the trial
                #print(f"Trial {i}: Correct trial with {len(saccade_indices)} saccades, first saccade speed: {saccade_speeds_correct[0]:.2f} deg/frame")
                #trial_success[i] = 2  # Mark this trial as succeincorrect before correct
            else:
                saccade_speeds_correct_first_saccade_all.append(saccade_speeds_correct[0])  # first saccade in the trial
                saccade_latency_correct_first_saccade_all.append((saccades['saccade_frames_xy'][saccade_indices[0]] - f)/60.0)  # latency of the first saccade in the trial
    elif trial_success[i] == -1: ## Incorrect trial
        # Find the number of saccade between go frame and end of the trial frame
        saccade_indices = np.where((saccades['saccade_frames_xy'] >= f) & (saccades['saccade_frames_xy'] <= end_of_trial_frame[i]))[0]
        if len(saccade_indices) > 0:
            saccade_speeds_incorrect = np.linalg.norm(saccades['eye_vel'][saccades['saccade_indices_xy'][saccade_indices]], axis=1)
            saccade_directions_incorrect = np.rad2deg(np.arctan2(saccades['eye_vel'][saccades['saccade_indices_xy'][saccade_indices], 1], saccades['eye_vel'][saccades['saccade_indices_xy'][saccade_indices], 0]))
            saccade_speeds_incorrect_all.append(saccade_speeds_incorrect[0])
            saccade_speeds_incorrect_latency_all.append((saccades['saccade_frames_xy'][saccade_indices[0]] - f)/60)  # latency of the first saccade in the trial
# saccade_speeds_correct_all = np.array(saccade_speeds_correct_all)
# saccade_speeds_correct_latency_all = np.array(saccade_speeds_correct_latency_all)
# saccade_speeds_incorrect_before_correct_all = np.array(saccade_speeds_incorrect_before_correct_all)
# saccade_speeds_incorrect_all = np.array(saccade_speeds_incorrect_all)
# saccade_speeds_incorrect_latency_all = np.array(saccade_speeds_incorrect_latency_all)
# saccade_speeds_correct_first_saccade_all = np.array(saccade_speeds_correct_first_saccade_all)
# saccade_latency_correct_first_saccade_all = np.array(saccade_latency_correct_first_saccade_all)
print(f"Number of correct trials with saccades: {len(saccade_speeds_correct_all)}, Number of incorrect trials with saccades: {len(saccade_speeds_incorrect_all)}")
print(f"Number of correct trials with both incorrect and correct saccades: {len(saccade_speeds_incorrect_before_correct_all)}")
print(f"Number of correct trials with only one correct saccade: {len(saccade_speeds_correct_first_saccade_all)}")
# Plot the box plot for saccade speeds for correct, incorrect, incorrect before correct, and correct first saccade trials
fig, ax = plt.subplots(figsize=(10, 6))
data = [saccade_speeds_correct_all, saccade_speeds_incorrect_all, saccade_speeds_incorrect_before_correct_all, saccade_speeds_correct_first_saccade_all]
labels = ['Correct Trials', 'Incorrect Trials', 'Incorrect Before Correct', 'Correct First Saccade']
ax.boxplot(data, labels=labels, patch_artist=True,
           boxprops=dict(facecolor='lightgreen', color='green'),    
              medianprops=dict(color='red'))    
ax.set_ylabel('Saccade Speed (deg/frame)')
ax.set_title('Comparison of Saccade Speeds by Trial Outcome')
#ax.grid(axis='y', alpha='0.3')
plt.tight_layout()
plt.show()
# Plot the box plot for saccade latencies for correct, incorrect, and incorrect before correct  and correct first saccade trials
fig, ax = plt.subplots(figsize=(10, 6))
data = [saccade_speeds_correct_latency_all, saccade_speeds_incorrect_latency_all, saccade_latency_incorrect_before_correct_all, saccade_latency_correct_first_saccade_all]
labels = ['Correct Trials', 'Incorrect Trials', 'Incorrect Before Correct', 'Correct First Saccade']
ax.boxplot(data, labels=labels, patch_artist=True,  
              boxprops=dict(facecolor='lightgreen', color='green'),
                medianprops=dict(color='red'))
ax.set_ylabel('Saccade Latency (s)')
ax.set_title('Comparison of Saccade Latencies by Trial Outcome')
#ax.grid(axis='y', alpha='0.3')
plt.tight_layout()
plt.show()



In [5]:
def calculate_entropy(data, bins=40):
    counts, _ = np.histogram(data, bins=bins, density=False)
    p = counts / counts.sum()
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

### Create a function that returns the entropy and variance of the accelerometer data for the whole sessions
def calculate_entropy_variance_of_accleometer(sessions_path):
        pattern_x = re.compile(r"acc[a-zA-Z]*_x$")
        pattern_y = re.compile(r"acc[a-zA-Z]*_y$")
        pattern_z = re.compile(r"acc[a-zA-Z]*_z$")
        ## A dictionary to hold avg entropy of imu acclereometer data for each session
        avg_entropy_accelerometer = {}
        avg_variance_accelerometer = {}
        for folder in os.listdir(sessions_path):
                folder_path = os.path.join(sessions_path, folder)
                if os.path.isdir(folder_path):
                        imu_files = [f for f in os.listdir(folder_path) if f.endswith('.csv') and 'IMU' in f]
                        if len(imu_files) > 0:
                                ## Read the imu file and check if the data in the  time column increases linearly
                                imu_file = os.path.join(folder_path, imu_files[0])
                                imu_data = pd.read_csv(imu_file)
        
                                # if acc*x,(can be acc_x or accel_x) use replace that with acc_x, same for acc*y and acc*z
                                imu_data.columns = [re.sub(pattern_x, 'acc_x', col) for col in imu_data.columns]
                                imu_data.columns = [re.sub(pattern_y, 'acc_y', col) for col in imu_data.columns]
                                imu_data.columns = [re.sub(pattern_z, 'acc_z', col) for col in imu_data.columns]
                                #print(f"Column names in {imu_file}: {imu_data.columns}")
                                #Check if the column name is 'Time' or 'time' 
                                if 'time' in imu_data.columns:
                                        time_data = imu_data['time'].values
                                elif 'Time' in imu_data.columns:    
                                        time_data = imu_data['Time'].values
                                else:
                                        #print(f"Warning: No time column found in {imu_file}. Skipping this file.")
                                        continue
                                # Check if time data increases linearly
                                time_diff = np.diff(time_data)
                                if np.all(time_diff > 0):
                                        #print(f"Time data in {imu_file} increases linearly.")
                                        # Lets check the entropy in the accelerometer data to see if the data is good
                                        acc_x = imu_data['acc_x'].values
                                        acc_y = imu_data['acc_y'].values
                                        acc_z = imu_data['acc_z'].values
                                        ## Calculate the entropy of the accelerometer data
                                        entropy_x = calculate_entropy(acc_x)
                                        entropy_y = calculate_entropy(acc_y)
                                        entropy_z = calculate_entropy(acc_z)
                                        avg_entropy = (entropy_x + entropy_y + entropy_z) / 3

                                        if avg_entropy > 0:  # Check if the average entropy is greater than 0
                                                avg_entropy_accelerometer[folder_path] = avg_entropy
                                                variance_x = np.var(acc_x)
                                                variance_y = np.var(acc_y)
                                                variance_z = np.var(acc_z)
                                                avg_variance = (variance_x + variance_y + variance_z) / 3
                                                avg_variance_accelerometer[folder_path] = avg_variance
                                        # if avg_entropy < 1:  # Check if the average entropy is less than 1, which might indicate bad data
                                        #         print(f"Warning: Average entropy of accelerometer data in {imu_file} is very low ({avg_entropy:.2f}). This might indicate bad data.")
                                        else:
                                                continue       
        return avg_entropy_accelerometer, avg_variance_accelerometer

In [6]:
##Lets select a folder from gui 
#sessions_path = select_folder()
all_session_paths = [
    "X:\Experimental_Data\EyeHeadCoupling_RatTS_server\Rat15_Too_server",
    "X:\Experimental_Data\EyeHeadCoupling_RatTS_server\Rat17_Anise_server",
    "X:\Experimental_Data\EyeHeadCoupling_RatTS_server\Rat22_Bayleaf_server",
    "X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server",
    "X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh02_Apollo_server",
]

## Create a csv file to save the average entropy and variance of the accelerometer data for each session


with open("average_entropy_variance.csv", "w+") as f:  
    f.write("Session_Path,Average_Entropy,Average_Variance\n")
    for session_path in all_session_paths:
        avg_entropy_accelerometer, avg_variance_accelerometer = calculate_entropy_variance_of_accleometer(session_path)
        for keys in avg_entropy_accelerometer.keys():
            f.write(f"{keys},{avg_entropy_accelerometer[keys]},{avg_variance_accelerometer[keys]}\n")

# ### Plot the average entropy of the accelerometer data for all sessions
# if (0):
#     fig, ax = plt.subplots(figsize=(10, 6))
#     ax.bar(range(len(avg_entropy_accelerometer)), avg_entropy_accelerometer, color='blue', edgecolor='k')
#     ax.set_xlabel('Session Index')
#     ax.set_ylabel('Average Entropy of Accelerometer Data')
#     ax.set_title('Average Entropy of Accelerometer Data Across Sessions')
#     # Plot the histogram of the average entropy values

#     fig, ax = plt.subplots(figsize=(10, 6))
#     ax.hist(avg_entropy_accelerometer, bins=10, color='blue', alpha=0.7)
#     ax.set_xlabel('Average Entropy of Accelerometer Data')
#     ax.set_ylabel('Number of Sessions')
#     ax.set_title('Histogram of Average Entropy of Accelerometer Data Across Sessions')

#     ## Plot the histogram of the average variance values (limit to 0.01)
#     fig, ax = plt.subplots(figsize=(10, 6))
#     ax.hist(list(avg_variance_accelerometer.values()), bins=10, color='orange', alpha=0.7, range=(0, 0.001))
#     ax.set_xlabel('Average Variance of Accelerometer Data')
#     ax.set_ylabel('Number of Sessions')
#     ax.set_title('Histogram of Average Variance of Accelerometer Data Across Sessions')
#     plt.show()


C:\Users\rp672\AppData\Local\Temp\ipykernel_105888\3426012216.py:3: RuntimeWarning: invalid value encountered in divide
  p = counts / counts.sum()
C:\Users\rp672\AppData\Local\Temp\ipykernel_105888\3426012216.py:3: RuntimeWarning: invalid value encountered in divide
  p = counts / counts.sum()
C:\Users\rp672\AppData\Local\Temp\ipykernel_105888\3426012216.py:3: RuntimeWarning: invalid value encountered in divide
  p = counts / counts.sum()
C:\Users\rp672\AppData\Local\Temp\ipykernel_105888\3426012216.py:3: RuntimeWarning: invalid value encountered in divide
  p = counts / counts.sum()
C:\Users\rp672\AppData\Local\Temp\ipykernel_105888\3426012216.py:3: RuntimeWarning: invalid value encountered in divide
  p = counts / counts.sum()
C:\Users\rp672\AppData\Local\Temp\ipykernel_105888\3426012216.py:3: RuntimeWarning: invalid value encountered in divide
  p = counts / counts.sum()
C:\Users\rp672\AppData\Local\Temp\ipykernel_105888\3426012216.py:3: RuntimeWarning: invalid value encountered in

In [ ]:
imu_f=pd.read_csv("X:\Experimental_Data\EyeHeadCoupling_RatTS_server\Rat17_Anise_server\Rat017_2024-12-19T10_34_05_closedloop\Rat017_IMU_2024-12-19T10_34_05.csv")
# Plot acc_x, acc_y, and acc_z over time for this file
fig, ax = plt.subplots(figsize=(12, 6))
time_data = imu_f['Time'].values
acc_x = imu_f['acc_x'].values
acc_y = imu_f['acc_y'].values
acc_z = imu_f['acc_z'].values
ax.plot(time_data, acc_x, label='acc_x', color='blue')
ax.plot(time_data, acc_y, label='acc_y', color='orange')
ax.plot(time_data, acc_z, label='acc_z', color='green')
ax.set_xlabel('Time')
ax.set_ylabel('Accelerometer Values')
ax.set_title('Accelerometer Data Over Time')



Text(0.5, 1.0, 'Accelerometer Data Over Time')

In [8]:
## 
csv_file = "C:\\Users\\rp672\\Documents\\EyeHeadCoupling\\Python\\notebooks\\average_entropy_variance.csv"
data = pd.read_csv(csv_file)
## Split session path into session name and folder 
for index, row in data.iterrows():
    session_path = row['Session_Path']
    session_name = os.path.basename(session_path)
    data.at[index, 'name'] = session_name
    # Add a new column for the folder name (parent folder of the session folder)
    folder_name = os.path.dirname(session_path)
    data.at[index, 'folder'] = folder_name
## DUmp it into a csv file 
data.to_csv("average_entropy_variance_with_names.csv", index=False)

In [10]:
## Population stats of the number of saccades across sessions in rats and treeshrews 
population_stats_csv = "C:\\Users\\rp672\\Documents\\EyeHeadCoupling\\MATLAB\\Figure2_EyeHead\\analysis_final_testingplotting_RP\\file_database_stats.csv"
population_stats = pd.read_csv(population_stats_csv)
## Drop rows with less than 70% dlc_percent_good_l 
population_stats = population_stats[population_stats['dlc_percent_good_l'] >= 0.8]
#population_stats['saccades_per_minute'] = population_stats['num_saccades'] / (population_stats['session_duration']) 
## Plot the number of saccades normalized by the duraction across all sessions for rats and treeshrews
fig, ax = plt.subplots(figsize=(10, 6))
total_rat_saccades = population_stats[population_stats['name'].str.contains('Rat')]['num_saccades'].sum()
total_rat_duration = population_stats[population_stats['name'].str.contains('Rat')]['session_duration'].sum()
total_treeshrew_saccades = population_stats[population_stats['name'].str.contains('Tsh')]['num_saccades'].sum()
total_treeshrew_duration = population_stats[population_stats['name'].str.contains('Tsh')]['session_duration'].sum()
rat_saccades_per_minute = total_rat_saccades / total_rat_duration
treeshrew_saccades_per_minute = total_treeshrew_saccades / total_treeshrew_duration
ax.bar(['Rats', 'Treeshrews'], [rat_saccades_per_minute, treeshrew_saccades_per_minute], color=['blue', 'orange'], edgecolor='k',width=0.8)
ax.set_ylabel('Saccades per Minute')
ax.set_title('Average Saccades per Minute Across Sessions')
# Print on the figure the total number of saccades and total duration for rats and treeshrews
# For shrews print it inside the bar, for rats print it above the bar
ax.text(0, rat_saccades_per_minute + 0.1, f'Number of Rats: 3\nTotal Saccades: {total_rat_saccades}\nTotal Duration: {total_rat_duration:.2f} min', ha='center', va='bottom')
ax.text(1, treeshrew_saccades_per_minute / 2, f'Number of Treeshrews: 2\nTotal Saccades: {total_treeshrew_saccades}\nTotal Duration: {total_treeshrew_duration:.2f} min', ha='center', va='center', color='black')
plt.tight_layout()
plt.show()

## Plot the population stats for head movements 
fig, ax = plt.subplots(figsize=(10, 6))
total_rat_hm_bouts = population_stats[population_stats['name'].str.contains('Rat')]['hm_bouts'].sum()
total_rat_hm_duration = population_stats[population_stats['name'].str.contains('Rat')]['hm_durations_sec'].sum()
total_treeshrew_hm_bouts = population_stats[population_stats['name'].str.contains('Tsh')]['hm_bouts'].sum()
total_treeshrew_hm_duration = population_stats[population_stats['name'].str.contains('Tsh')]['hm_durations_sec'].sum()
rat_hm_bouts_per_minute = total_rat_hm_bouts / total_rat_duration
treeshrew_hm_bouts_per_minute = total_treeshrew_hm_bouts / total_treeshrew_duration
ax.bar(['Rats', 'Treeshrews'], [rat_hm_bouts_per_minute, treeshrew_hm_bouts_per_minute], color=['blue', 'orange'], edgecolor='k',width=0.8)
ax.set_ylabel('Head Movement Bouts per Minute')
ax.set_title('Average Head Movement Bouts per Minute Across Sessions')
# Print on the figure the total number of head movement bouts and total duration for rats and treeshrews
# For shrews print it inside the bar, for rats print it inside the bar
ax.text(0, rat_hm_bouts_per_minute / 2, f'Number of Rats: 3\nTotal HM Bouts: {total_rat_hm_bouts}\nTotal Duration: {total_rat_duration:.2f} min', ha='center', va='center', color='black')
ax.text(1, treeshrew_hm_bouts_per_minute / 2, f'Number of Treeshrews: 2\nTotal HM Bouts: {total_treeshrew_hm_bouts}\nTotal Duration: {total_treeshrew_duration:.2f} min', ha='center', va='center', color='black')
plt.tight_layout()
plt.show()

## Plot the population stats for total durations of head movements across sessions in rats and treeshrews
fig, ax = plt.subplots(figsize=(10, 6))
total_rat_hm_duration = population_stats[population_stats['name'].str.contains('Rat')]['hm_durations_sec'].sum()/60.0  # convert to minutes
total_treeshrew_hm_duration = population_stats[population_stats['name'].str.contains('Tsh')]['hm_durations_sec'].sum()/60.0  # convert to minutes
rat_hm_duration_per_minute = total_rat_hm_duration / total_rat_duration
treeshrew_hm_duration_per_minute = total_treeshrew_hm_duration / total_treeshrew_duration
ax.bar(['Rats', 'Treeshrews'], [rat_hm_duration_per_minute, treeshrew_hm_duration_per_minute], color=['blue', 'orange'], edgecolor='k',width=0.8)
ax.set_ylabel('Head Movement Duration per Minute')
ax.set_title('Average Head Movement Duration per Minute Across Sessions')
# Print on the figure the total number of head movement durations and total duration for rats and treeshrews
# For shrews print it inside the bar, for rats print it inside the bar
ax.text(0, rat_hm_duration_per_minute / 2, f'Number of Rats: 3\nTotal HM Duration: {total_rat_hm_duration:.2f} min\nTotal Duration: {total_rat_duration:.2f} min', ha='center', va='center', color='black')
ax.text(1, treeshrew_hm_duration_per_minute / 2, f'Number of Treeshrews: 2\nTotal HM Duration: {total_treeshrew_hm_duration:.2f} min\nTotal Duration: {total_treeshrew_duration:.2f} min', ha='center', va='center', color='black')
plt.tight_layout()
plt.show()

In [11]:
population_stats_csv = "C:\\Users\\rp672\\Documents\\EyeHeadCoupling\\MATLAB\\Figure2_EyeHead\\analysis_final_testingplotting_RP\\file_database_stats.csv"
population_stats = pd.read_csv(population_stats_csv)
population_stats = population_stats[population_stats['dlc_percent_good_l'] >= 0.6]

# Per-session rates
population_stats['saccades_per_minute']  = population_stats['num_left_saccades']    / (population_stats['session_duration'] * population_stats['dlc_percent_good_l'])
population_stats['hm_bouts_per_minute']  = population_stats['hm_bouts']        / (population_stats['session_duration'] * population_stats['dlc_percent_good_l'])
population_stats['hm_duration_per_minute'] = (population_stats['hm_durations_sec'] / 60.0) / (population_stats['session_duration'] * population_stats['dlc_percent_good_l'])

rats      = population_stats[population_stats['name'].str.contains('Rat')]
treeshrews = population_stats[population_stats['name'].str.contains('Tsh')]

def box_with_points(ax, data_dict, ylabel, title, colors):
    labels = list(data_dict.keys())
    values = list(data_dict.values())

    bp = ax.boxplot(values,
                    positions=range(len(labels)),
                    widths=0.4,
                    patch_artist=True,
                    showfliers=False,
                    medianprops=dict(color='black', linewidth=2))

    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.4)

    for i, (vals, color) in enumerate(zip(values, colors)):
        jitter = np.random.uniform(-0.08, 0.08, size=len(vals))
        ax.scatter(i + jitter, vals, color=color, edgecolors='k',
                   s=60, zorder=3, linewidths=0.8, alpha = 0.2)

    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12)

    # Annotate n per group
    # for i, (label, vals) in enumerate(zip(labels, values)):
    #     ax.text(i, ax.get_ylim()[1] * 0.97,
    #             f'n={len(vals)} sessions',
    #             ha='center', va='top', fontsize=8, color='gray')

colors = ['#4C72B0', '#DD8452']  # blue, orange

# --- Saccades per minute ---
fig, ax = plt.subplots(figsize=(6, 5))
box_with_points(ax,
    {'Rats': rats['saccades_per_minute'].dropna().values,
     'Treeshrews': treeshrews['saccades_per_minute'].dropna().values},
    ylabel='Saccades per Minute',
    title='Saccades per Minute per Session',
    colors=colors)
plt.tight_layout()
plt.show()

# --- HM bouts per minute ---
fig, ax = plt.subplots(figsize=(6, 5))
box_with_points(ax,
    {'Rats': rats['hm_bouts_per_minute'].dropna().values,
     'Treeshrews': treeshrews['hm_bouts_per_minute'].dropna().values},
    ylabel='Head Movement Bouts per Minute',
    title='Head Movement Bouts per Minute per Session',
    colors=colors)
plt.tight_layout()
plt.show()

# --- HM duration per minute ---
fig, ax = plt.subplots(figsize=(6, 5))
box_with_points(ax,
    {'Rats': rats['hm_duration_per_minute'].dropna().values,
     'Treeshrews': treeshrews['hm_duration_per_minute'].dropna().values},
    ylabel='Head Movement Duration per Minute (min/min)',
    title='Head Movement Duration per Minute per Session',
    colors=colors)
plt.tight_layout()
plt.show()

In [19]:
###Read the csv file from Erin's analysis
sac_imu_pop = pd.read_csv("C:\\Users\\rp672\\Documents\\EyeHeadCoupling\\MATLAB\\Figure2_EyeHead\\saccade_imu_subset.csv")

## Separate the data into Rats and Treeshrews
eyehead_coupling_rats = sac_imu_pop[sac_imu_pop['animal_id'].str.contains('Rat', na=False)]
eyehead_coupling_treeshrews = sac_imu_pop[sac_imu_pop['animal_id'].str.contains('Tsh', na=False)]


head_rats = np.sort(eyehead_coupling_rats['max_std_window'].to_numpy())
head_treeshrews = np.sort(eyehead_coupling_treeshrews['max_std_window'].to_numpy())
cdf_rats = np.arange(1, len(head_rats)+1) / len(head_rats)
cdf_treeshrews = np.arange(1, len(head_treeshrews)+1) / len(head_treeshrews)

threshold = 2


frac_rats = (head_rats <= threshold).sum() / len(head_rats)
frac_tsh  = (head_treeshrews <= threshold).sum() / len(head_treeshrews)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(head_rats, cdf_rats, label='Rats', color='blue')
ax.plot(head_treeshrews, cdf_treeshrews, label='Treeshrews', color='orange')

# vertical threshold line
ax.axvline(threshold, color='red', linestyle='--', )
ax.axvline(3, color='red', linestyle='--', )

ax.set_xlabel('Max accelerometer signal during saccade (Noise SD)')
ax.set_ylabel('Cumulative Probability')
ax.set_title('CDF of Max Accelerometer Signal During Saccades')
ax.set_xscale('log', base=2)
ax.set_xlim(0.5, 2**5)

ticks = [1, 2, 4, 8, 16, 32]
ax.set_xticks(ticks)
ax.xaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())

# annotate percentages (left = frac, right = 1-frac)
ax.text(threshold, 0.2,
        f'Rats: {frac_rats*100:.1f}% ≤ 2\nRats: {(1-frac_rats)*100:.1f}% > 2',
        color='blue', ha='left', va='bottom')

ax.text(threshold, 0.4,
        f'Treeshrews: {frac_tsh*100:.1f}% ≤ 2\nTreeshrews: {(1-frac_tsh)*100:.1f}% > 2',
        color='orange', ha='left', va='bottom')
frac_tsh_3 = (head_treeshrews <= 3).sum() / len(head_treeshrews)
frac_rats_3 = (head_rats <= 3).sum() / len(head_rats)
ax.text(3, 0.6,
        f'Treeshrews: {frac_tsh_3*100:.1f}% ≤ 3\nTreeshrews: {(1-frac_tsh_3)*100:.1f}% > 3',
        color='orange', ha='left', va='bottom')
ax.text(3, 0.8,
        f'Rats: {frac_rats_3*100:.1f}% ≤ 3\nRats: {(1-frac_rats_3)*100:.1f}% > 3',
        color='blue', ha='left', va='bottom')

ax.legend()
plt.show()




In [20]:
## Plot categorical bar graph of the fraction of rats and treeshrews with max accelerometer signal and without during saccades with multiple thresholds (1,1.5,2). With accleremeter is with attempted head movements
thresholds = [1, 1.5, 2, 2.5]
frac_rats_with = []
frac_rats_without = []
frac_tsh_with = []
frac_tsh_without = []
for threshold in thresholds:
    frac_rats_with.append((head_rats > threshold).sum() / len(head_rats))
    frac_rats_without.append((head_rats <= threshold).sum() / len(head_rats))
    frac_tsh_with.append((head_treeshrews > threshold).sum() / len(head_treeshrews))
    frac_tsh_without.append((head_treeshrews <= threshold).sum() / len(head_treeshrews))
x = np.arange(len(thresholds))
width = 0.3
fig, ax = plt.subplots(figsize=(10, 6))
# Plot only the "without attempted head movements" bars
ax.bar(x - width/2, frac_rats_without, width, label='Rats', color='blue')
ax.bar(x + width/2, frac_tsh_without, width, label='Treeshrews', color='orange')
ax.set_xlabel('Accelerometer Threshold (Noise SD)')
ax.set_ylabel('Fraction of Saccades')
ax.set_title('Fraction of saccades without attempted head movements in rats and treeshrews')
ax.legend()
## Set the tick labels to the thresholds
ax.set_xticks(x)
ax.set_xticklabels([str(t) for t in thresholds])
# Print the percentage value inside the bars
for i in range(len(thresholds)):
    ax.text(x[i] - width/2, frac_rats_without[i]/2, f'{frac_rats_without[i]*100:.1f}%', color='black', ha='center', va='center')
    ax.text(x[i] + width/2, frac_tsh_without[i]/2, f'{frac_tsh_without[i]*100:.1f}%', color='black', ha='center', va='center')

In [15]:
thresholds = [2, 3]
frac_rats_with    = []
frac_rats_without = []
frac_tsh_with     = []
frac_tsh_without  = []

for threshold in thresholds:
    frac_rats_with.append(   (head_rats        > threshold).sum() / len(head_rats))
    frac_rats_without.append((head_rats        <= threshold).sum() / len(head_rats))
    frac_tsh_with.append(    (head_treeshrews  > threshold).sum() / len(head_treeshrews))
    frac_tsh_without.append( (head_treeshrews  <= threshold).sum() / len(head_treeshrews))

fig, axes = plt.subplots(1, 2, figsize=(10, 6), sharey=True)

groups        = ['Rats', 'Treeshrews']
fracs_with    = [frac_rats_with,    frac_tsh_with]
fracs_without = [frac_rats_without, frac_tsh_without]
colors        = ['#4C72B0', '#DD8452']

x     = np.arange(2)   # two bars per subplot (one per threshold)
width = 0.5

for ax, thresh_idx, threshold in zip(axes, range(len(thresholds)), thresholds):
    f_without = [frac_rats_without[thresh_idx], frac_tsh_without[thresh_idx]]
    f_with    = [frac_rats_with[thresh_idx],    frac_tsh_with[thresh_idx]]

    ax.bar(x, f_without, width, label='Without HM',
           color=colors, alpha=0.4, edgecolor='k')
    ax.bar(x, f_with,    width, label='With HM',
           color=colors, alpha=0.9, edgecolor='k', bottom=f_without)

    # Labels inside segments
    for i in range(2):
        if f_without[i] > 0.05:
            ax.text(x[i], f_without[i] / 2,
                    f'{f_without[i]*100:.1f}%',
                    ha='center', va='center', fontsize=11, color='black')
        if f_with[i] > 0.05:
            ax.text(x[i], f_without[i] + f_with[i] / 2,
                    f'{f_with[i]*100:.1f}%',
                    ha='center', va='center', fontsize=11, color='white')

    ax.set_xticks(x)
    ax.set_xticklabels(groups, fontsize=11)
    ax.set_title(f'Threshold = {threshold}x noise', fontsize=12)
    ax.set_xlabel('Animal group', fontsize=10)

# Legend on first subplot only
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='gray', alpha=0.4, edgecolor='k', label='Without HM'),
                   Patch(facecolor='gray', alpha=0.9, edgecolor='k', label='With HM')]
axes[0].legend(handles=legend_elements, fontsize=9, loc='lower right')
axes[0].set_ylabel('Fraction of Saccades', fontsize=11)

plt.suptitle('Fraction of saccades with/without attempted head movements', fontsize=13)
plt.tight_layout()
plt.show()

C:\Users\rp672\AppData\Local\Temp\ipykernel_148200\1384746212.py:8: RuntimeWarning: invalid value encountered in scalar divide
  frac_rats_with.append(   (head_rats        > threshold).sum() / len(head_rats))
C:\Users\rp672\AppData\Local\Temp\ipykernel_148200\1384746212.py:9: RuntimeWarning: invalid value encountered in scalar divide
  frac_rats_without.append((head_rats        <= threshold).sum() / len(head_rats))


In [20]:
## Plot scatter plot of max_std and sac_mag for rats and tree shrews
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(eyehead_coupling_rats['sac_mag'], eyehead_coupling_rats['max_std_window'], label='Rats', color='blue', alpha=0.7)
ax.scatter(eyehead_coupling_treeshrews['sac_mag'], eyehead_coupling_treeshrews['max_std_window'], label='Treeshrews', color='orange', alpha=0.7)
ax.set_xlabel('Saccade Magnitude (deg)')
ax.set_ylabel('Max Accelerometer Signal During Saccade (Noise SD)')
ax.set_ylim(0.5, 32)
ax.set_title('Max Accelerometer Signal During Saccades vs Saccade Magnitude')

Text(0.5, 1.0, 'Max Accelerometer Signal During Saccades vs Saccade Magnitude')

In [ ]:


# def identify_saccades_2d(position_xy, velocity_xy, acceleration_xy,
#                          velocity_threshold=60,
#                          acceleration_threshold=1000,
#                          window_size=75,
#                          min_duration=16,
#                          fixation_velocity_threshold=10):
#     """
#     position_xy:    (N, 2) array [x,y] in deg (or px)
#     velocity_xy:    (N, 2) array [vx,vy] in deg/s
#     acceleration_xy:(N, 2) array [ax,ay] in deg/s^2

#     Returns:
#         saccades:                 (K,) indices of saccade peaks (time index)
#         saccade_windows:          (K,2) start,end indices for each saccade
#         saccade_durations:        (K,) durations in samples
#         saccade_amplitudes:       (K,2) 2D displacement [dx,dy]
#         saccade_amplitudes_norm:  (K,) scalar amplitude (norm of [dx,dy])
#         saccade_peak_velocities:  (K,) peak speed (deg/s) within each saccade
#     """

#     speed = np.linalg.norm(velocity_xy, axis=1)
#     acc_mag = np.linalg.norm(acceleration_xy, axis=1)

#     saccades = []
#     saccade_windows = []
#     saccade_durations = []
#     saccade_amplitudes = []
#     saccade_amplitudes_norm = []
#     saccade_peak_velocities = []

#     # Candidate peaks in speed
#     peaks, _ = find_peaks(np.abs(speed),
#                           height=velocity_threshold,
#                           distance=min_duration)

#     n = len(speed)

#     for peak in peaks:
#         # Local window (clipped)
#         start = max(0, peak - window_size)
#         end   = min(n - 1, peak + window_size)

#         # Acceleration check
#         if np.max(acc_mag[start:end+1]) <= acceleration_threshold:
#             continue

#         # Refine start: backward until speed is below fixation threshold
#         s = start
#         for i in range(peak, start, -1):
#             if speed[i] < fixation_velocity_threshold:
#                 s = i
#                 break

#         # Refine end: forward until speed is below fixation threshold
#         e = end
#         for i in range(peak, end):
#             if speed[i] < fixation_velocity_threshold:
#                 e = i
#                 break

#         # 2D amplitude
#         delta = position_xy[e] - position_xy[s]
#         amp_norm = np.linalg.norm(delta)

#         # Peak speed within this saccade window
#         peak_speed = np.max(speed[s:e+1])

#         saccades.append(peak)
#         saccade_windows.append((s, e))
#         saccade_durations.append(e - s)
#         saccade_amplitudes.append(delta)
#         saccade_amplitudes_norm.append(amp_norm)
#         saccade_peak_velocities.append(peak_speed)

#     return (np.array(saccades),
#             np.array(saccade_windows),
#             np.array(saccade_durations),
#             np.array(saccade_amplitudes),
#             np.array(saccade_amplitudes_norm),
#             np.array(saccade_peak_velocities))


In [13]:
from scipy.optimize import curve_fit
def _logistic_quad_model(t, a0, a1, a2, A, w):
    """
    Logistic + quadratic trend, 5 parameters.

    t : time (relative to window center)
    f(t) = a0 + a1*t + a2*t^2 + A * logistic(t/w)
    """
    tau = t
    sigma = 1.0 / (1.0 + np.exp(-tau / w))
    return a0 + a1*tau + a2*tau**2 + A * sigma

def _spline4_fit(t, y):
    """
    Fit 4th-order polynomial (5 params) to y(t).
    Returns predicted y and SSE.
    """
    tau = t
    M = np.column_stack([
        np.ones_like(tau),
        tau,
        tau**2,
        tau**3,
        tau**4
    ])
    coeffs, *_ = np.linalg.lstsq(M, y, rcond=None)
    y_hat = M @ coeffs
    sse = np.sum((y - y_hat)**2)
    return y_hat, sse

def _fit_logistic_vs_spline(time, pos_xy, peak_idx,
                            window_ms=150.0,
                            improvement_thresh=0.5):
    """
    For one candidate saccade peak, fit logistic vs spline models
    in a 150 ms window; return refined start/end indices and success flag.

    time    : (N,) time in seconds
    pos_xy  : (N,2) 2D position (x,y)
    peak_idx: index of candidate saccade (int)
    """
    dt = np.median(np.diff(time))
    fs = 1.0 / dt

    half_win = int(round(0.5 * window_ms / 1000.0 * fs))
    n = len(time)

    start = max(0, peak_idx - half_win)
    end   = min(n - 1, peak_idx + half_win)

    t_win = time[start:end+1]
    x_win = pos_xy[start:end+1, 0]
    y_win = pos_xy[start:end+1, 1]

    # Center time within window
    t_center = 0.5 * (t_win[0] + t_win[-1])
    t_rel = t_win - t_center

    # --- Fit logistic+quad and spline to X and Y independently ---
    # X dimension
    y = x_win
    p0_log_x = [np.mean(y), 0.0, 0.0, (y[-1] - y[0]), (t_rel[-1] - t_rel[0]) / 10.0 or 1e-3]
    try:
        popt_log_x, _ = curve_fit(_logistic_quad_model, t_rel, y,
                                  p0=p0_log_x, maxfev=5000)
    except Exception:
        return None, None, False  # fit failed
    y_log_x = _logistic_quad_model(t_rel, *popt_log_x)
    sse_log_x = np.sum((y - y_log_x)**2)
    _, sse_spl_x = _spline4_fit(t_rel, y)

    # Y dimension
    y = y_win
    p0_log_y = [np.mean(y), 0.0, 0.0, (y[-1] - y[0]), (t_rel[-1] - t_rel[0]) / 10.0 or 1e-3]
    try:
        popt_log_y, _ = curve_fit(_logistic_quad_model, t_rel, y,
                                  p0=p0_log_y, maxfev=5000)
    except Exception:
        return None, None, False
    y_log_y = _logistic_quad_model(t_rel, *popt_log_y)
    sse_log_y = np.sum((y - y_log_y)**2)
    _, sse_spl_y = _spline4_fit(t_rel, y)

    # Total SSE over both dimensions
    sse_log_tot = sse_log_x + sse_log_y
    sse_spl_tot = sse_spl_x + sse_spl_y

    # Require logistic to be at least 50% better than spline
    improvement = (sse_spl_tot - sse_log_tot) / sse_spl_tot
    if improvement < improvement_thresh:
        return None, None, False

    # Use logistic width parameters to define start/end
    w_x = popt_log_x[4]
    w_y = popt_log_y[4]
    w = 0.5 * (abs(w_x) + abs(w_y))  # average width

    t_start = t_center - 3.0 * w
    t_end   = t_center + 3.0 * w

    # Map times back to indices in the full trace
    idx_start_rel = np.argmin(np.abs(t_win - t_start))
    idx_end_rel   = np.argmin(np.abs(t_win - t_end))
    idx_start = start + idx_start_rel
    idx_end   = start + idx_end_rel

    # Ensure ordering
    if idx_end <= idx_start:
        return None, None, False

    return idx_start, idx_end, True


In [88]:

def identify_saccades_3d(position_xyz, velocity_xyz, acceleration_xyz, time,
                         velocity_thresholds=(60, 60, 60),
                         acceleration_thresholds=(3000, 3000, 3000),
                         window_size=30,
                         min_duration=20,
                         fixation_velocity_thresholds=(40, 40, 40),
                         use_logistic_refinement=False,
                         logistic_improvement_thresh=0.5,
                         debug=True):
    """
    3D saccade detection: horizontal, vertical, torsion.

    position_xyz:       (N, 3) [x,y,z] in deg (or px)
    velocity_xyz:       (N, 3) [vx,vy,vz] in deg/s
    acceleration_xyz:   (N, 3) [ax,ay,az] in deg/s^2
    time:               (N,)   seconds

    Returns
    -------
    union_saccades:           (K,)   indices of saccade peaks (global)
    union_saccade_windows:    (K,2)  [start,end] indices (global)
    union_saccade_durations:  (K,)   samples
    union_saccade_amp_norm:   (K,)   3D amplitude magnitude |[dx,dy,dz]|
    union_saccade_peak_speed: (K,)   peak 3D speed (norm of v)

    per_axis_saccades:        list of 3 arrays, each (Ki,)
    per_axis_saccade_windows: list of 3 arrays, each (Ki,2)
    per_axis_saccade_dur:     list of 3 arrays, each (Ki,)
    per_axis_saccade_amp:     list of 3 arrays, each (Ki,)  magnitude on that axis
    per_axis_peak_speed:      list of 3 arrays, each (Ki,)  peak speed on that axis
    """

    pos = np.asarray(position_xyz)
    vel = np.asarray(velocity_xyz)
    acc = np.asarray(acceleration_xyz)
    time = np.asarray(time)

    fs = 1.0 / np.median(np.diff(time))
    n = len(time)

    per_axis_saccades = []
    per_axis_windows = []
    per_axis_durations = []
    per_axis_amplitudes = []      # magnitude per axis
    per_axis_peak_speed = []

    # -------- per-axis detection --------
    for axis in range(3):
        v = vel[:, axis]
        a = acc[:, axis]
        speed_axis = np.abs(v)

        v_thr   = velocity_thresholds[axis]
        a_thr   = acceleration_thresholds[axis]
        fix_thr = fixation_velocity_thresholds[axis]

        saccades = []
        saccade_windows = []
        saccade_durations = []
        saccade_amplitudes = []        # magnitude on this axis
        saccade_peak_velocities = []

        peaks, _ = find_peaks(speed_axis,
                              height=v_thr,
                              distance=min_duration)
        acc_mag_axis = np.abs(a)

        for peak in peaks:
            start0 = max(0, peak - window_size)
            end0   = min(n - 1, peak + window_size)

            # acceleration gate
            if np.max(acc_mag_axis[start0:end0+1]) <= a_thr:
                continue

            if use_logistic_refinement:
                s, e, ok = _fit_logistic_vs_spline(
                    time, pos[:, [axis]], peak,
                    window_ms=150.0,
                    improvement_thresh=logistic_improvement_thresh,
                )
                if not ok:
                    continue
            else:
                # backward search: stop at fixation or local minimum
                s = peak
                for i in range(peak - 1, start0 + 1, -1):
                    if speed_axis[i] <= fix_thr:
                        s = i
                        break
                    if speed_axis[i-1] > speed_axis[i] < speed_axis[i+1]:
                        s = i
                        break

                # forward search
                e = peak
                for i in range(peak + 1, end0 - 1):
                    if speed_axis[i] <= fix_thr:
                        e = i
                        break
                    if speed_axis[i-1] > speed_axis[i] < speed_axis[i+1]:
                        e = i
                        break

                if e <= s:
                    continue

                if speed_axis[s] > fix_thr or speed_axis[e] > fix_thr:
                    continue

                dur = e - s
                min_dur_samples = int(0.02 * fs)   # 20 ms
                max_dur_samples = int(0.10 * fs)   # 100 ms
                if not (min_dur_samples <= dur <= max_dur_samples):
                    continue

                # avoid merging multiple strong peaks on THIS axis
                if np.any((peaks > s) & (peaks < e) & (peaks != peak)):
                    continue

            # amplitude magnitude along this axis only
            amp_axis = abs(pos[e, axis] - pos[s, axis])
            peak_speed_axis = np.max(speed_axis[s:e+1])

            saccades.append(peak)
            saccade_windows.append((s, e))
            saccade_durations.append(e - s)
            saccade_amplitudes.append(amp_axis)
            saccade_peak_velocities.append(peak_speed_axis)

        per_axis_saccades.append(np.array(saccades, dtype=int))
        per_axis_windows.append(np.array(saccade_windows, dtype=int))
        per_axis_durations.append(np.array(saccade_durations, dtype=int))
        per_axis_amplitudes.append(np.array(saccade_amplitudes, dtype=float))
        per_axis_peak_speed.append(np.array(saccade_peak_velocities, dtype=float))

    # -------- union (3D) saccades from all axes --------
    all_windows = []
    for aw in per_axis_windows:
        if aw.size > 0:
            all_windows.extend(list(map(tuple, aw)))

    if len(all_windows) == 0:
        union_saccades = np.array([], dtype=int)
        union_windows  = np.empty((0, 2), dtype=int)
        union_durs     = np.array([], dtype=int)
        union_amp_norm = np.array([], dtype=float)
        union_peak3d   = np.array([], dtype=float)

        return (union_saccades, union_windows, union_durs,
                union_amp_norm, union_peak3d,
                per_axis_saccades, per_axis_windows, per_axis_durations,
                per_axis_amplitudes, per_axis_peak_speed)

    # sort and merge overlapping windows
    all_windows = sorted(all_windows, key=lambda w: w[0])
    merged = []
    cur_s, cur_e = all_windows[0]
    for s, e in all_windows[1:]:
        if s <= cur_e:      # overlap
            cur_e = max(cur_e, e)
        else:
            merged.append((cur_s, cur_e))
            cur_s, cur_e = s, e
    merged.append((cur_s, cur_e))
    merged = np.array(merged, dtype=int)

    # compute 3D features for union saccades
    union_saccades = []
    union_windows = []
    union_durations = []
    union_amp_norm = []
    union_peak3d = []

    for (s, e) in merged:
        if e <= s:
            continue

        v_seg = vel[s:e+1]                     # (T,3)
        speed_total = np.linalg.norm(v_seg, axis=1)
        peak_local = np.argmax(speed_total)
        peak_idx = s + peak_local

        delta = pos[e] - pos[s]
        amp_norm = np.linalg.norm(delta)
        peak_speed3d = speed_total.max()

        union_saccades.append(peak_idx)
        union_windows.append((s, e))
        union_durations.append(e - s)
        union_amp_norm.append(amp_norm)
        union_peak3d.append(peak_speed3d)

    union_saccades = np.array(union_saccades, dtype=int)
    union_windows  = np.array(union_windows, dtype=int)
    union_durations = np.array(union_durations, dtype=int)
    union_amp_norm = np.array(union_amp_norm, dtype=float)
    union_peak3d = np.array(union_peak3d, dtype=float)

    return (union_saccades, union_windows, union_durations,
            union_amp_norm, union_peak3d,
            per_axis_saccades, per_axis_windows, per_axis_durations,
            per_axis_amplitudes, per_axis_peak_speed)


In [21]:
def identify_saccades_1d(position, velocity, time,
                         window_size=30,
                         min_duration_ms=20,
                         max_duration_ms=100,
                         mad_multiplier=6.0,
                         fixation_mad_multiplier=2.0,   # lower than saccade multiplier
                         mad_max_iter=20,
                         mad_tolerance=1e-3,
                         use_logistic_refinement=False,
                         logistic_improvement_thresh=0.5):
    """
    1D saccade detection with fully data-driven velocity and fixation thresholds.

    Both thresholds derived from the same MAD estimation:
      - saccade threshold  = median + mad_multiplier         * MAD
      - fixation threshold = median + fixation_mad_multiplier * MAD

    Parameters
    ----------
    position                  : (N,) position in deg or px
    velocity                  : (N,) velocity in deg/s
    time                      : (N,) time in seconds
    window_size               : int, search window half-width (samples)
    min_duration_ms           : float, minimum saccade duration in ms
    max_duration_ms           : float, maximum saccade duration in ms
    mad_multiplier            : scalar, saccade threshold multiplier (default 6)
    fixation_mad_multiplier   : scalar, fixation threshold multiplier (default 2)
    mad_max_iter              : int, maximum iterations (default 20)
    mad_tolerance             : float, convergence tolerance (default 1e-3)
    use_logistic_refinement   : bool
    logistic_improvement_thresh : float

    Returns
    -------
    saccades                  : (K,)   peak indices
    saccade_windows           : (K, 2) [start, end] indices
    saccade_durations         : (K,)   duration in samples
    saccade_amplitudes        : (K,)   amplitude (abs position change)
    saccade_peak_speeds       : (K,)   peak speed
    velocity_threshold        : float  converged saccade threshold
    fixation_threshold        : float  converged fixation threshold
    """

    pos   = np.asarray(position).ravel()
    vel   = np.asarray(velocity).ravel()
    time  = np.asarray(time).ravel()

    fs    = 1.0 / np.median(np.diff(time))
    n     = len(time)
    speed = np.abs(vel)

    min_dur_samples = int(min_duration_ms / 1000.0 * fs)
    max_dur_samples = int(max_duration_ms / 1000.0 * fs)

    # ----------------------------------------------------------------
    # MAD-based iterative threshold estimation
    # ----------------------------------------------------------------
    baseline_mask      = np.ones(n, dtype=bool)
    velocity_threshold = np.inf

    for iteration in range(mad_max_iter):
        baseline_speed = speed[baseline_mask]

        med = np.median(baseline_speed)
        mad = np.median(np.abs(baseline_speed - med))

        new_threshold = med + mad_multiplier * mad

        if abs(new_threshold - velocity_threshold) < mad_tolerance:
            velocity_threshold = new_threshold
            break

        velocity_threshold = new_threshold
        baseline_mask      = speed <= velocity_threshold

        if baseline_mask.sum() < 0.1 * n:
            break

    # Fixation threshold from same MAD estimate
    fixation_threshold = med + fixation_mad_multiplier * mad

    # ----------------------------------------------------------------
    # Saccade detection
    # ----------------------------------------------------------------
    saccades            = []
    saccade_windows     = []
    saccade_durations   = []
    saccade_amplitudes  = []
    saccade_peak_speeds = []

    peaks, _ = find_peaks(speed,
                          height=velocity_threshold,
                          distance=min_dur_samples)

    for peak in peaks:
        start0 = max(0, peak - window_size)
        end0   = min(n - 1, peak + window_size)

        if use_logistic_refinement:
            pos_2d = pos[:, np.newaxis]
            s, e, ok = _fit_logistic_vs_spline(
                time, pos_2d, peak,
                window_ms=150.0,
                improvement_thresh=logistic_improvement_thresh,
            )
            if not ok:
                continue
        else:
            # Backward search
            s = peak
            for i in range(peak - 1, start0 + 1, -1):
                if speed[i] <= fixation_threshold:
                    s = i
                    break
                if speed[i-1] > speed[i] < speed[i+1]:
                    s = i
                    break

            # Forward search
            e = peak
            for i in range(peak + 1, end0 - 1):
                if speed[i] <= fixation_threshold:
                    e = i
                    break
                if speed[i-1] > speed[i] < speed[i+1]:
                    e = i
                    break

            if e <= s:
                continue
            if speed[s] > fixation_threshold or \
               speed[e] > fixation_threshold:
                continue

            dur = e - s
            if not (min_dur_samples <= dur <= max_dur_samples):
                continue

            if np.any((peaks > s) & (peaks < e) & (peaks != peak)):
                continue

        amp        = abs(pos[e] - pos[s])
        peak_speed = np.max(speed[s:e+1])

        saccades.append(peak)
        saccade_windows.append((s, e))
        saccade_durations.append(e - s)
        saccade_amplitudes.append(amp)
        saccade_peak_speeds.append(peak_speed)

    return (np.array(saccades,            dtype=int),
            np.array(saccade_windows,     dtype=int),
            np.array(saccade_durations,   dtype=int),
            np.array(saccade_amplitudes,  dtype=float),
            np.array(saccade_peak_speeds, dtype=float),
            velocity_threshold,
            fixation_threshold)

In [ ]:
## Plot the main sequence per session
'''Steps: 1. Find the left eye file that has run through DLC
2. Get the 8 points around pupil and 4 eye points that DLC has tracked 
3. Calulcate the pupil center by fitting an ellipse to the 8 points around the pupil and get the center of the ellipse
4. Calculate the eye position in degrees by using the center of the pupil and the 4 eye points and using the eye model to convert to degrees
5. DO median filtering of the eye position data to remove noise
6. Upsampple the eye position data to 1 khz by using interpolation
7. Butterworth low pass filter the eye position data to remove high frequency noise
8. Calculate the eye velocity by taking the derivative of the eye position data
9. Use the eye velocity data to detect saccades by using a threshold velocity and accleration
10. Calculate the saccade magnitude by taking the difference of the eye position before and after the saccade
11. Plot the main sequence by plotting the saccade magnitude vs saccade velocity for each session
'''

def extract_saccades_from_folder(folder_path, cal, 
                                 time_start=None,
                                 time_end = None,
                                 fps=60.0,
                                 new_fs=500.0,
                                 lp_cutoff=25.0,
                                 pupil_likelihood_thresh=0.95,
                                 eye_likelihood_thresh=0.70,
                                 plot=True):
    """
    From a folder containing a DLC left-eye CSV, compute saccade velocities
    and magnitudes (in deg) using your current pipeline.
    Returns:
        saccade_velocities (1D array, deg/s)
        saccade_magnitudes (1D array, deg)
        saccade_durations (1D array, samples)
        saccade_windows (2D array, start, end indices)
        new_time (1D array, time in seconds)
        eye_position_filtered (2D array, x,y in deg)
        eye_velocity (2D array, x,y in deg/s)
        acceleration (2D array, x,y in deg/s^2)
    """

    # --- Step 1: find DLC left-eye file ---
    DLC_left_eye_files = [
        f for f in os.listdir(folder_path)
        if f.endswith('.csv') and 'DLC' in f and 'E_L' in f
    ]
    if len(DLC_left_eye_files) == 0:
        raise FileNotFoundError(f"No DLC left eye file found in {folder_path}")
    DLC_left_eye_file = DLC_left_eye_files[0]

    # --- Step 2: load DLC CSV and extract points ---
    csv_path = os.path.join(folder_path, DLC_left_eye_file)
    DLC_data = pd.read_csv(csv_path, skiprows=2)

    pupil_points = DLC_data[
        ['x.4', 'y.4', 'x.5', 'y.5', 'x.6', 'y.6', 'x.7', 'y.7',
         'x.8', 'y.8', 'x.9', 'y.9', 'x.10', 'y.10', 'x.11', 'y.11']
    ].values

    pupil_points_likelihood = DLC_data[
        ['likelihood.4', 'likelihood.5', 'likelihood.6', 'likelihood.7',
         'likelihood.8', 'likelihood.9', 'likelihood.10', 'likelihood.11']
    ].values

    eye_points = DLC_data[['x', 'y', 'x.2', 'y.2']].values
    eye_points_likelihood = DLC_data[['likelihood', 'likelihood.2']].values

    eye_points_VD = DLC_data[['x.1', 'y.1', 'x.3', 'y.3']].values
    eye_points_VD_likelihood = DLC_data[['likelihood.1', 'likelihood.3']].values

    n_frames = pupil_points.shape[0]

    # --- Step 3: pupil centers via ellipse fit ---
    pupil_centers = []
    for i in range(n_frames):
        pts = pupil_points[i].reshape(-1, 2).astype(np.float32)
        if np.any(pts == 0):  # missing points
            pupil_centers.append([np.nan, np.nan, np.nan])
            continue
        # Check likelihood of each point
        likelihoods = pupil_points_likelihood[i]
        ## If atleast 6 points have good likelihood, we can fit an ellipse, otherwise we skip this frame
        n_good_points = np.sum(likelihoods >= pupil_likelihood_thresh)
        if n_good_points < 8:
            pupil_centers.append([np.nan, np.nan, np.nan])
            continue
        ellipse = cv2.fitEllipse(pts)
        center = ellipse[0]   # (cx, cy)
        angle = ellipse[2]    # angle of the ellipse in degrees
        # Store center and angle. Angle will be used for torsion estimation 
        pupil_centers.append([center[0], center[1], angle])
    pupil_centers = np.array(pupil_centers, dtype=float)

    #print(pupil_centers)

    # --- Step 3b: eye centers (mean of 2 eye points) ---
    eye_centers = []
    for i in range(n_frames):
        pts = eye_points[i].reshape(-1, 2).astype(np.float32)
        if np.any(pts == 0):
            eye_centers.append([np.nan, np.nan])
            continue
        # Check likelihood of each point
        likelihoods = eye_points_likelihood[i]
        if np.any(likelihoods < eye_likelihood_thresh):  # if any point has low likelihood, skip
            eye_centers.append([np.nan, np.nan])
            continue
        center = np.mean(pts, axis=0)
        eye_centers.append(center)
    eye_centers = np.array(eye_centers, dtype=float)

    eye_VD_distance = []
    for i in range(n_frames):
        pts = eye_points_VD[i].reshape(-1, 2).astype(np.float32)
        if np.any(pts == 0):
            eye_VD_distance.append(np.nan)
            continue
        # Check likelihood of each point
        likelihoods = eye_points_VD_likelihood[i]
        if np.any(likelihoods < eye_likelihood_thresh):  # if any point has low likelihood, skip
            eye_VD_distance.append(np.nan)
            continue
        center = np.mean(pts, axis=0)
        eye_VD_distance.append(np.linalg.norm(pts[0] - pts[1]))
    eye_VD_distance = np.array(eye_VD_distance, dtype=float)

    # --- Step 3c: interpolate missing pupil and eye centers ---

    eye_VD_distance = pd.Series(eye_VD_distance).interpolate().bfill().ffill()

    # --- Step 3d: Blink detection and interpolation (optional) ---
    # Based on the velocity of the eye_VD_centers, we can detect blinks as periods of high velocity and interpolate over them
    # DIstance between the two eye_VD points
    eye_VD_velocity = np.abs(np.gradient(eye_VD_distance))
    blink_thresh = np.percentile(eye_VD_velocity, 99.99) # e.g. top 0.01% of velocities
    # Mark the blink period when the velocity exceeds threshold and the eye or eye_VD_distance is below a certain threshold (indicating the eye is closed)
    eye_VD_distance_thresh = np.percentile(eye_VD_distance, 5)
    blink_indices = np.where((eye_VD_velocity > blink_thresh) )[0]
    for idx in blink_indices:
        start = max(0, idx - 5)  # 5 frames before
        end = min(n_frames - 1, idx + 5)  # 5 frames after
        pupil_centers[start:end+1] = np.nan
        eye_centers[start:end+1] = np.nan
    for j in range(2):  # x and y
        pupil_centers[:, j] = pd.Series(pupil_centers[:, j]).interpolate().bfill().ffill()
        eye_centers[:, j] = pd.Series(eye_centers[:, j]).interpolate().bfill().ffill()
    pupil_centers[:, 2] = pd.Series(pupil_centers[:, 2]).interpolate().bfill().ffill()  # angle for torsion
    # ## Plot the vdistance and velocity with the blink indices shaded
    # if plot:
    #     fig, ax = plt.subplots(figsize=(12, 6))
    #     time = DLC_data['coords'].values / fps
    #     ax.plot(time, eye_VD_distance, label='Eye VD Distance')
    #     ax.plot(time, eye_VD_velocity, label='Eye VD Velocity')
    #     for idx in blink_indices:
    #         ax.axvspan(time[max(0, idx - 5)], time[min(n_frames - 1, idx + 5)], color='red', alpha=0.2)
    #     ax.axhline(eye_VD_distance_thresh, color='green', linestyle='--', label='Eye VD Distance Threshold')
    #     ax.axhline(blink_thresh, color='black', linestyle='--', label='Blink Velocity Threshold')
    #     ax.set_xlabel('Time (s)')
    #     ax.set_title('Eye VD Distance and Velocity with Blinks Highlighted')
    #     ax.legend()
    #     plt.show()

    # Do an exponential moving average of the eye_centers to smooth them out, which will help with the eye position estimation
    eye_centers[:, 0] = pd.Series(eye_centers[:, 0]).ewm(span=20).mean().values
    eye_centers[:, 1] = pd.Series(eye_centers[:, 1]).ewm(span=20).mean().values


    # --- Step 4: eye position in degrees (relative pupil–eye center / cal) ---
    eye_positions = np.array(pupil_centers)[:, :2] - eye_centers     # pixels
    #eye_positions = pupil_centers     # pixels
    eye_position_in_degrees = eye_positions / cal   # deg
    ## Add the third dimension to the eye position, which is torsion (already in degrees from the ellipse fit)
    eye_position_in_degrees = np.hstack((eye_position_in_degrees, pupil_centers[:, 2:3]))  # deg

    # --- Step 5: median filter per component ---
    eye_position_in_degrees[:, 0] = medfilt(eye_position_in_degrees[:, 0], kernel_size=3)
    eye_position_in_degrees[:, 1] = medfilt(eye_position_in_degrees[:, 1], kernel_size=3)
    eye_position_in_degrees[:, 2] = medfilt(eye_position_in_degrees[:, 2], kernel_size=3)
    # --- Step 6: upsample to new_fs (e.g. 1 kHz) ---
    original_time = DLC_data['coords'].values / fps
    t_end = original_time[-1]
    new_dt = 1.0 / new_fs
    new_time = np.arange(0, t_end, new_dt)

    interp_func_x = interp1d(original_time, eye_position_in_degrees[:, 0],
                             kind='linear', fill_value='extrapolate')
    interp_func_y = interp1d(original_time, eye_position_in_degrees[:, 1],
                             kind='linear', fill_value='extrapolate')
    eye_x = interp_func_x(new_time)
    eye_y = interp_func_y(new_time)
    eye_z = interp1d(original_time, eye_position_in_degrees[:, 2], kind='linear', fill_value='extrapolate')(new_time)
    eye_position_interp = np.vstack((eye_x, eye_y, eye_z)).T

    # --- Step 7: Butterworth low-pass filter ---
    nyq = 0.5 * new_fs
    b, a = butter(2, lp_cutoff / nyq, btype='low')
    eye_x_f = filtfilt(b, a, eye_position_interp[:, 0])
    eye_y_f = filtfilt(b, a, eye_position_interp[:, 1])
    eye_z_f = filtfilt(b, a, eye_position_interp[:, 2])
    eye_position_filtered = np.vstack((eye_x_f, eye_y_f, eye_z_f)).T

    # --- Step 8: velocity (gradient wrt time) ---
    eye_velocity_x = np.gradient(eye_position_filtered[:, 0], new_time)
    eye_velocity_y = np.gradient(eye_position_filtered[:, 1], new_time)
    eye_velocity_z = np.gradient(eye_position_filtered[:, 2], new_time)
    eye_velocity = np.vstack((eye_velocity_x, eye_velocity_y, eye_velocity_z)).T

    # # --- Step 8b: velocity filtering (optional) ---
    # eye_velocity_x_f = filtfilt(b, a, eye_velocity[:, 0])
    # eye_velocity_y_f = filtfilt(b, a, eye_velocity[:, 1])
    # eye_velocity = np.vstack((eye_velocity_x_f, eye_velocity_y_f)).T

    # --- Step 9: acceleration and saccade detection ---
    accel_x = np.gradient(eye_velocity[:, 0], new_time)
    accel_y = np.gradient(eye_velocity[:, 1], new_time)
    accel_z = np.gradient(eye_velocity[:, 2], new_time)
    acceleration = np.vstack((accel_x, accel_y, accel_z)).T

    speed = np.linalg.norm(eye_velocity, axis=1)
    acc_mag = np.linalg.norm(acceleration, axis=1)

    # Optional: If time_start and time_end are provided, restrict to that time window
    if time_start is not None and time_end is not None:
        time_mask = (new_time >= time_start) & (new_time <= time_end)
        eye_position_filtered = eye_position_filtered[time_mask]
        eye_velocity = eye_velocity[time_mask]
        acceleration = acceleration[time_mask]
        new_time = new_time[time_mask]
        speed = speed[time_mask]
        acc_mag = acc_mag[time_mask]

    # saccade_indices = np.where(
    #     (speed > vel_thresh) & (acc_mag > acc_thresh)
    # )[0]
    ## Use the identify_saccades function to get saccade indices
    (saccade_indices, saccade_windows, saccade_durations,
     saccade_amplitudes_norm, saccade_peak_velocities,
     per_axis_saccades, per_axis_windows, per_axis_durations,
     per_axis_amplitudes, per_axis_peak_speed) = identify_saccades_3d(
        eye_position_filtered, eye_velocity, acceleration, new_time)

    ## Plot the x,y position, velcoity,and accleration with shaded regions indicating the saccade indices on tthe same time axis
    if plot:
        fig, axs = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
        axs[0].plot(new_time, eye_position_filtered[:, 0], label='Eye X (deg)')
        axs[0].plot(new_time, eye_position_filtered[:, 1], label='Eye Y (deg)')
        axs[0].plot(new_time, eye_position_filtered[:, 2], label='Eye Z (deg)')
        axs[0].set_ylabel('Position (deg)')
        axs[0].legend()
        for start, end in saccade_windows:
            axs[0].axvspan(new_time[start], new_time[end], color='red', alpha=0.2)

        # ## Plot the un filtered pupil center and eye center positions to see how they look before the processing steps
        # axs[0].plot(original_time, pupil_centers[:, 0], label='Pupil Center X (px)', color='cyan', alpha=0.5)
        # axs[0].plot(original_time, pupil_centers[:, 1], label='Pupil Center Y (px)', color='magenta', alpha=0.5)
        # axs[0].plot(original_time, eye_centers[:, 0], label='Eye Center X (px)', color='green', alpha=0.5)
        # axs[0].plot(original_time, eye_centers[:, 1], label='Eye Center Y (px)', color='yellow', alpha=0.5)
        # #axs[0].plot(original_time, eye_centers[:, 2], label='Eye Center Z (px)', color='orange', alpha=0.5)
        # axs[0].legend()

        axs[1].plot(new_time, eye_velocity[:, 0], label='Velocity X (deg/s)')
        axs[1].plot(new_time, eye_velocity[:, 1], label='Velocity Y (deg/s)')
        axs[1].plot(new_time, eye_velocity[:, 2], label='Velocity Z (deg/s)')
        axs[1].set_ylabel('Velocity (deg/s)')
        axs[1].legend()
        for start, end in saccade_windows:
            axs[1].axvspan(new_time[start], new_time[end], color='red', alpha=0.2)

        axs[2].plot(new_time, acceleration[:, 0], label='Acceleration X (deg/s^2)')
        axs[2].plot(new_time, acceleration[:, 1], label='Acceleration Y (deg/s^2)')
        axs[2].plot(new_time, acceleration[:, 2], label='Acceleration Z (deg/s^2)')
        axs[2].set_ylabel('Acceleration (deg/s^2)')
        axs[2].legend()
        for start, end in saccade_windows:
            axs[2].axvspan(new_time[start], new_time[end], color='red', alpha=0.2)

        plt.tight_layout()
        plt.show()

 

    return saccade_peak_velocities, saccade_amplitudes_norm, saccade_durations, saccade_windows, new_time, eye_position_filtered, eye_velocity, acceleration, per_axis_durations, per_axis_amplitudes, per_axis_peak_speed

    





In [31]:
import sys, os
import cv2
import numpy as np
import matplotlib
matplotlib.use("QtAgg")

from PySide6 import QtCore, QtGui, QtWidgets
from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure


class SaccadeViewer(QtWidgets.QMainWindow):
    def __init__(self, video_path, time, pos, vel, acc,
                 saccade_windows,
                 saccade_amplitudes_norm=None,
                 saccade_peak_velocities=None,
                 parent=None):
        super().__init__(parent)
        self.setWindowTitle("Saccade visualizer")

        # ---- store data ----
        self.time = time
        self.pos = pos
        self.vel = vel
        self.acc = acc
        self.saccade_windows = np.asarray(saccade_windows)
        self.saccade_amplitudes_norm = np.asarray(saccade_amplitudes_norm) \
            if saccade_amplitudes_norm is not None else None
        self.saccade_peak_velocities = np.asarray(saccade_peak_velocities) \
            if saccade_peak_velocities is not None else None

        # video
        self.cap = cv2.VideoCapture(video_path)
        self.fps = self.cap.get(cv2.CAP_PROP_FPS)
        self.n_frames = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # current time index and current saccade index
        self.idx = 0
        self.current_sac = 0 if len(self.saccade_windows) > 0 else None

        # ---- central widget ----
        central = QtWidgets.QWidget()
        self.setCentralWidget(central)

        main_layout = QtWidgets.QHBoxLayout(central)

        # left side: video + plots
        left = QtWidgets.QVBoxLayout()
        main_layout.addLayout(left, stretch=4)

        # video label
        self.video_label = QtWidgets.QLabel()
        self.video_label.setMinimumHeight(300)
        left.addWidget(self.video_label)

        # matplotlib canvas
        self.fig = Figure(figsize=(6, 4))
        self.canvas = FigureCanvas(self.fig)
        left.addWidget(self.canvas)

        self.ax_pos = self.fig.add_subplot(3, 1, 1)
        self.ax_vel = self.fig.add_subplot(3, 1, 2, sharex=self.ax_pos)
        self.ax_acc = self.fig.add_subplot(3, 1, 3, sharex=self.ax_pos)

        # pre-create lines
        t = self.time
        self.line_posx, = self.ax_pos.plot(t, self.pos[:, 0], label="X (deg)")
        self.line_posy, = self.ax_pos.plot(t, self.pos[:, 1], label="Y (deg)")
        self.cursor_pos = self.ax_pos.axvline(0, color="r")

        self.line_velx, = self.ax_vel.plot(t, self.vel[:, 0], label="X vel")
        self.line_vely, = self.ax_vel.plot(t, self.vel[:, 1], label="Y vel")
        self.cursor_vel = self.ax_vel.axvline(0, color="r")

        self.line_accx, = self.ax_acc.plot(t, self.acc[:, 0], label="X acc")
        self.line_accy, = self.ax_acc.plot(t, self.acc[:, 1], label="Y acc")
        self.cursor_acc = self.ax_acc.axvline(0, color="r")

        self.ax_pos.legend()
        self.ax_vel.legend()
        self.ax_acc.legend()
        self.ax_acc.set_xlabel("Time (s)")

        # shade saccade windows
        for s, e in self.saccade_windows:
            ts, te = self.time[s], self.time[e]
            for ax in (self.ax_pos, self.ax_vel, self.ax_acc):
                ax.axvspan(ts, te, color="orange", alpha=0.2)

        self.fig.tight_layout()

        # right side: info labels
        right = QtWidgets.QVBoxLayout()
        main_layout.addLayout(right, stretch=1)

        self.info_label = QtWidgets.QLabel()
        self.info_label.setAlignment(QtCore.Qt.AlignTop | QtCore.Qt.AlignLeft)
        self.info_label.setText(
            "Controls:\n"
            "Space: next saccade\n"
            "Shift+Space: previous saccade\n"
        )
        right.addWidget(self.info_label)

        self.saccade_label = QtWidgets.QLabel()
        self.saccade_label.setAlignment(QtCore.Qt.AlignTop | QtCore.Qt.AlignLeft)
        right.addWidget(self.saccade_label)

        right.addStretch(1)

        # initial draw
        self.update_to_current_saccade()

    # ---- keyboard control ----
    def keyPressEvent(self, event):
        key = event.key()
        modifiers = event.modifiers()

        # Space: next saccade
        if key == QtCore.Qt.Key_Space and modifiers == QtCore.Qt.NoModifier:
            if self.current_sac is not None and self.current_sac < len(self.saccade_windows) - 1:
                self.current_sac += 1
                self.update_to_current_saccade()
            return

        # Shift+Space: previous saccade
        if key == QtCore.Qt.Key_Space and modifiers & QtCore.Qt.ShiftModifier:
            if self.current_sac is not None and self.current_sac > 0:
                self.current_sac -= 1
                self.update_to_current_saccade()
            return

        super().keyPressEvent(event)

    def update_to_current_saccade(self):
        if self.current_sac is None or len(self.saccade_windows) == 0:
            return
        s, e = self.saccade_windows[self.current_sac]
        self.idx = int((s + e) / 2)  # center of saccade
        self.update_all()

        # update text about this saccade
        t0 = self.time[s]
        t1 = self.time[e]
        amp = (self.saccade_amplitudes_norm[self.current_sac]
               if self.saccade_amplitudes_norm is not None else np.nan)
        vpeak = (self.saccade_peak_velocities[self.current_sac]
                 if self.saccade_peak_velocities is not None else np.nan)
        self.saccade_label.setText(
            f"Saccade {self.current_sac+1}/{len(self.saccade_windows)}\n"
            f"Time: {t0:.3f}–{t1:.3f} s\n"
            f"Amplitude: {amp:.2f} deg\n"
            f"Peak velocity: {vpeak:.1f} deg/s"
        )

    # ---- update video + plots ----
    def update_all(self):
        t_current = self.time[self.idx]
        for cursor in (self.cursor_pos, self.cursor_vel, self.cursor_acc):
            cursor.set_xdata([t_current, t_current])
        self.canvas.draw_idle()

        frame_idx = int(round(t_current * self.fps))
        frame_idx = max(0, min(self.n_frames - 1, frame_idx))
        self.cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = self.cap.read()
        if not ok:
            return
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        h, w, ch = frame_rgb.shape
        bytes_per_line = ch * w
        qimg = QtGui.QImage(frame_rgb.data, w, h, bytes_per_line,
                            QtGui.QImage.Format_RGB888)
        pix = QtGui.QPixmap.fromImage(qimg)
        self.video_label.setPixmap(
            pix.scaled(self.video_label.size(),
                       QtCore.Qt.KeepAspectRatio,
                       QtCore.Qt.SmoothTransformation)
        )


def launch_gui(video_path, time, pos, vel, acc,
               saccade_windows,
               saccade_amplitudes_norm,
               saccade_peak_velocities):
    app = QtWidgets.QApplication.instance()
    if app is None:
        app = QtWidgets.QApplication(sys.argv)
    win = SaccadeViewer(video_path, time, pos, vel, acc,
                        saccade_windows,
                        saccade_amplitudes_norm,
                        saccade_peak_velocities)
    win.resize(1100, 800)
    win.show()
    app.exec()


In [ ]:
folder = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\TSh01_Paris_server\Tsh001_2025-08-01T15_15_48"
#folder = r"X:\Experimental_Data\EyeHeadCoupling_RatTS_server\Rat17_Anise_server\Rat017_2024-11-07T15_16_29"
for f in os.listdir(folder):
    if f.endswith('.mp4') and 'E_L' in f and 'DLC' in f:
        video_path = os.path.join(folder, f)
        break

cal = 4.0  # example: pixels per degree (set from your calibration)
sacc_vel, sacc_mag, sacc_dur, sacc_win, new_time, eye_position_filtered, eye_velocity, acceleration, per_axis_durations, per_axis_amplitudes, per_axis_peak_speed = extract_saccades_from_folder(folder, cal,time_start=None, time_end=None, plot=False)

# launch_gui(
#     video_path="X:\\Experimental_Data\\EyeHeadCoupling_RatTS_server\\TSh01_Paris_server\\Tsh001_2025-08-01T15_15_48\\Tsh001_E_L_2025-08-01T15_15_49DLC_resnet50_EyeSep30shuffle1_2500000_labeled.mp4",
#     time=new_time,
#     pos=eye_position_filtered,
#     vel=eye_velocity,
#     acc=acceleration,
#     saccade_windows=sacc_win,
#     saccade_amplitudes_norm=sacc_mag,
#     saccade_peak_velocities=sacc_vel
# )


# Plot main sequence
fig, ax = plt.subplots(figsize=(10, 6))
#ax.scatter(per_axis_amplitudes[0], per_axis_peak_speed[0], color='blue', alpha=0.5,s=10, picker=True, pickradius=5)
#ax.scatter(per_axis_amplitudes[1], per_axis_peak_speed[1], color='red', alpha=0.5,s=10, picker=True, pickradius=5)
#ax.scatter(per_axis_amplitudes[2], per_axis_peak_speed[2], color='green', alpha=0.5,s=10, picker=True, pickradius=5)
ax.scatter(sacc_mag, sacc_vel, color='blue', alpha=0.5,s=10, picker=True, pickradius=5)
ax.set_xlabel('Saccade Magnitude (deg)')
ax.set_ylabel('Saccade Peak Velocity (deg/s)')
ax.set_title('Main Sequence: Saccade Peak Velocity vs Magnitude')



# fig, ax = plt.subplots(figsize=(10, 6))
# sacc_dur_sec = sacc_dur / 1000.0  # convert ms to seconds
# ax.scatter(sacc_mag,sacc_dur_sec, color='orange', alpha=0.5,s=10, picker=True, pickradius=5 )
# ax.set_xlabel('Saccade Magnitude (deg)')
# ax.set_ylabel('Saccade Duration (s)')

# fig, ax = plt.subplots(figsize=(10, 6))
# ax.scatter(sacc_vel,sacc_dur_sec, color='green', alpha=0.5,s=10, picker=True, pickradius=5)
# ax.set_xlabel('Saccade Peak Velocity (deg/s)')
# ax.set_ylabel('Saccade Duration (s)')
plt.show()

time = new_time




def on_pick(event):
    idx = event.ind[0]

    # --- existing trace plotting code (unchanged) ---
    s, e = sacc_win[idx]
    dur = e - s
    center = (s + e) // 2
    half_win = int(2 * dur)
    i0 = max(0, center - half_win)
    i1 = min(len(new_time) - 1, center + half_win)

    t_win   = new_time[i0:i1+1]
    pos_win = eye_position_filtered[i0:i1+1]
    vel_win = eye_velocity[i0:i1+1]
    acc_win = acceleration[i0:i1+1]

    fig_tr, axs = plt.subplots(3, 1, sharex=True, figsize=(8, 6))
    axs[0].plot(t_win, pos_win[:, 0], label='X')
    axs[0].plot(t_win, pos_win[:, 1], label='Y')
    axs[0].plot(t_win, pos_win[:, 2], label='Z (torsion)')
    axs[0].set_ylabel('Position (deg)')
    axs[0].legend()

    axs[1].plot(t_win, vel_win[:, 0], label='X')
    axs[1].plot(t_win, vel_win[:, 1], label='Y')
    axs[1].plot(t_win, vel_win[:, 2], label='Z (torsion)')
    axs[1].plot(t_win, np.linalg.norm(vel_win, axis=1),
                label='Speed', color='black', linestyle='--')
    axs[1].set_ylabel('Velocity (deg/s)')
    axs[1].legend()

    axs[2].plot(t_win, acc_win[:, 0], label='X')
    axs[2].plot(t_win, acc_win[:, 1], label='Y')
    axs[2].plot(t_win, acc_win[:, 2], label='Z (torsion)')
    axs[2].set_ylabel('Acceleration (deg/s²)')
    axs[2].set_xlabel('Time (s)')
    axs[2].legend()

    for ax_ in axs:
        ax_.axvspan(new_time[s], new_time[e], color='red', alpha=0.3)

    fig_tr.suptitle(
        f"Saccade {idx}: amp={sacc_mag[idx]:.2f} deg, "
        f"v_peak={np.linalg.norm(sacc_vel[idx]):.1f} deg/s"
    )
    plt.tight_layout()

    # --- non-blocking video player using a timer ---

    start_time = new_time[s] - 0.1
    end_time   = new_time[e] + 0.1

    video_fps = 60.0
    start_frame = max(0, int(round(start_time * video_fps)))
    end_frame   = int(round(end_time * video_fps))

    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    fig_video, ax_video = plt.subplots(figsize=(8, 6))
    ax_video.set_axis_off()
    ax_video.set_title("Space: play/pause, q: close")

    ret, frame = cap.read()
    if not ret:
        cap.release()
        plt.close(fig_video)
        return

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    im = ax_video.imshow(frame_rgb, animated=True)
    ts_text = ax_video.text(
        0.02, 0.95, "", color="yellow",
        fontsize=10, transform=ax_video.transAxes,
        ha="left", va="top", bbox=dict(facecolor='black', alpha=0.3, edgecolor='none')
    )
    paused = True
    current_frame = start_frame

    def on_key(event_k):
        nonlocal paused
        if event_k.key == ' ':
            paused = not paused
        elif event_k.key in ('q', 'escape'):
            # stop timer and close video window
            timer.stop()
            cap.release()
            plt.close(fig_video)

    fig_video.canvas.mpl_connect('key_press_event', on_key)

    def update_video(_): 
        nonlocal current_frame
        if paused:
            return
        ret, frame = cap.read()
        if (not ret) or (current_frame > end_frame):
            # loop
            cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
            current_frame = start_frame
            ret, frame = cap.read()
            if not ret:
                return
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        im.set_data(frame_rgb)

        # compute and show timestamp (in seconds)
        t_video = current_frame / video_fps
        ts_text.set_text(f"{t_video:.3f} s")

        fig_video.canvas.draw_idle()
        current_frame += 1

    # timer drives the video, no blocking loop
    timer = fig_video.canvas.new_timer(interval=int(1000 / video_fps))
    timer.add_callback(update_video, None)
    timer.start()





#plt.tight_layout()
#plt.show()

fig.canvas.mpl_connect('pick_event', on_pick)
plt.show()


: 

In [7]:
## Population  main sequence for rats and treeshrews
rat_folders = [ f for f in data['Session_Path'] if 'Rat' in f ]
treeshrew_folders = [ f for f in data['Session_Path'] if 'Tsh' in f ]
rat_sacc_velocities = []
rat_sacc_magnitudes = []
treeshrew_sacc_velocities = []
treeshrew_sacc_magnitudes = []
for folder in rat_folders:
    try:
        sacc_vel, sacc_mag = extract_saccades_from_folder(folder, cal)
        rat_sacc_velocities.extend(sacc_vel)
        rat_sacc_magnitudes.extend(sacc_mag)
    except Exception as e:
        print(f"Error processing {folder}: {e}")
for folder in treeshrew_folders:
    try:
        sacc_vel, sacc_mag = extract_saccades_from_folder(folder, cal)
        treeshrew_sacc_velocities.extend(sacc_vel)
        treeshrew_sacc_magnitudes.extend(sacc_mag)
    except Exception as e:
        print(f"Error processing {folder}: {e}")
# Plot population main sequence
# fig, ax = plt.subplots(figsize=(10, 6))
# ax.scatter(rat_sacc_magnitudes, rat_sacc_velocities, label='Rats', color='blue', alpha=0.7)
# ax.scatter(treeshrew_sacc_magnitudes, treeshrew_sacc_velocities, label='Treeshrews', color='orange', alpha=0.7)
# ax.set_xlabel('Saccade Magnitude (deg)')
# ax.set_ylabel('Saccade Velocity (deg/s)')
# ax.set_title('Population Main Sequence: Saccade Velocity vs Magnitude')
# ax.legend()
# plt.show()

NameError: name 'data' is not defined

In [75]:
## Main sequence for Tsh002
from scipy.optimize import curve_fit
Tsh01_folders = [f for f in data['Session_Path'] if 'Tsh002' in f]
Tsh01_sacc_velocities = []
Tsh01_sacc_magnitudes = []
for folder in Tsh01_folders:
    try:
        sacc_vel, sacc_mag, sacc_dur, sacc_win, new_time, eye_position_filtered, eye_velocity, acceleration = extract_saccades_from_folder(folder, cal, plot=False)
        Tsh01_sacc_velocities.extend(np.linalg.norm(sacc_vel, axis=1))
        Tsh01_sacc_magnitudes.extend(sacc_mag)
    except Exception as e:
        print(f"Error processing {folder}: {e}")

Tsh01_sacc_magnitudes = np.array(Tsh01_sacc_magnitudes)
Tsh01_sacc_velocities = np.array(Tsh01_sacc_velocities)
## Filter out saccades with magnitudes more than 20 degrees and speed more than 1000 deg/s, which are likely artifacts Also filter out saccades with magnitude less than 1 degree, which are likely noise
valid_indices = (Tsh01_sacc_magnitudes < 20) & (Tsh01_sacc_velocities < 1000) & (Tsh01_sacc_magnitudes > 1)
Tsh01_sacc_magnitudes = Tsh01_sacc_magnitudes[valid_indices]
Tsh01_sacc_velocities = Tsh01_sacc_velocities[valid_indices]

## Fit the data with y = (a * x )^b and plot the fitted curve
def main_sequence_func(x, a, b):
    return (a * x) ** b
popt, _ = curve_fit(main_sequence_func, Tsh01_sacc_magnitudes, Tsh01_sacc_velocities, p0=[100, 0.5])
# Plot main sequence for Tsh01
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(Tsh01_sacc_magnitudes, Tsh01_sacc_velocities, label='Tsh002', color='green', alpha=0.7, s=10)
x_fit = np.linspace(Tsh01_sacc_magnitudes.min(), Tsh01_sacc_magnitudes.max(), 100)
y_fit = main_sequence_func(x_fit, *popt)
ax.plot(x_fit, y_fit, label=f'Fitted: y = ({popt[0]:.2f} * x)^{popt[1]:.2f}', color='red')  
ax.set_xlabel('Saccade Magnitude (deg)')
ax.set_ylabel('Peak Velocity (deg/s)')
ax.set_title('Main Sequence for Tsh002: Peak Velocity vs Magnitude')
ax.legend()
plt.show()




C:\Users\rp672\AppData\Local\Temp\ipykernel_105888\269982479.py:23: RuntimeWarning: invalid value encountered in power
  return (a * x) ** b


In [46]:
## Main sequence for Rat017
Rat017_folders = [f for f in data['Session_Path'] if 'Rat022' in f]
Rat017_sacc_velocities = []
Rat017_sacc_magnitudes = []
for folder in Rat017_folders:
    try:
        sacc_vel, sacc_mag = extract_saccades_from_folder(folder, cal)
        Rat017_sacc_velocities.extend(sacc_vel)
        Rat017_sacc_magnitudes.extend(sacc_mag)
    except Exception as e:
        print(f"Error processing {folder}: {e}")
Rat017_sacc_magnitudes = np.array(Rat017_sacc_magnitudes)
Rat017_sacc_velocities = np.array(Rat017_sacc_velocities)   
## Filter out saccades with magnitudes more than 20 degrees and speed more than 1000 deg/s, which are likely artifacts
valid_indices = (Rat017_sacc_magnitudes < 15) & (Rat017_sacc_velocities < 1000)
Rat017_sacc_magnitudes = Rat017_sacc_magnitudes[valid_indices]
Rat017_sacc_velocities = Rat017_sacc_velocities[valid_indices]
## Fit the data with y = (a * x )^b and plot the fitted curve
popt, _ = curve_fit(main_sequence_func, Rat017_sacc_magnitudes, Rat017_sacc_velocities, p0=[100, 0.5])
# Plot main sequence for Rat017
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(Rat017_sacc_magnitudes, Rat017_sacc_velocities, label='Rat017', color='purple', alpha=0.7)
x_fit = np.linspace(Rat017_sacc_magnitudes.min(), Rat017_sacc_magnitudes.max(), 100)
y_fit = main_sequence_func(x_fit, *popt)
ax.plot(x_fit, y_fit, label=f'Fitted: y = ({popt[0]:.2f} * x)^{popt[1]:.2f}', color='red')
ax.set_xlabel('Saccade Magnitude (deg)')
ax.set_ylabel('Saccade Velocity (deg/s)')
ax.set_title('Main Sequence for Rat017: Saccade Velocity vs Magnitude')
ax.legend()
plt.show()

Error processing X:\Experimental_Data\EyeHeadCoupling_RatTS_server\Rat22_Bayleaf_server\Rat022_2025-06-03T12_24_04: No DLC left eye file found in X:\Experimental_Data\EyeHeadCoupling_RatTS_server\Rat22_Bayleaf_server\Rat022_2025-06-03T12_24_04


C:\Users\rp672\AppData\Local\Temp\ipykernel_101104\2268354729.py:23: RuntimeWarning: invalid value encountered in power
  return (a * x) ** b


In [10]:
## Plot the rat main sequence with a line fit
## Vet the data to remove outliers (saccades with magnitude > 20 deg or velocity > 1000 deg/s)
fig, ax = plt.subplots(figsize=(10, 6))
# Filter out outliers
filtered_magnitudes = []
filtered_velocities = []
for mag, vel in zip(rat_sacc_magnitudes, rat_sacc_velocities):
    if mag <= 20 and vel <= 1000:
        filtered_magnitudes.append(mag)
        filtered_velocities.append(vel)
ax.scatter(filtered_magnitudes, filtered_velocities, label='Rats', color='blue', alpha=0.7)
# Fit a line to the filtered rat data
coeffs = np.polyfit(filtered_magnitudes, filtered_velocities, deg=1)
fit_line = np.poly1d(coeffs)
x_fit = np.linspace(min(filtered_magnitudes), max(filtered_magnitudes), 100)
y_fit = fit_line(x_fit)
ax.plot(x_fit, y_fit, color='red', label='Linear Fit')
ax.set_xlabel('Saccade Magnitude (deg)')
ax.set_ylabel('Saccade Velocity (deg/s)')
ax.set_title('Rat Main Sequence with Linear Fit')
ax.legend()

In [11]:
## Plot the tree shrew main sequence with a line fit
fig, ax = plt.subplots(figsize=(10, 6))
# Filter out outliers
filtered_magnitudes = []
filtered_velocities = []
for mag, vel in zip(treeshrew_sacc_magnitudes, treeshrew_sacc_velocities):
    if mag <= 20 and vel <= 1000:
        filtered_magnitudes.append(mag)
        filtered_velocities.append(vel)
ax.scatter(filtered_magnitudes, filtered_velocities, label='Treeshrews', color='orange', alpha=0.7)
# Fit a line to the filtered treeshrew data
coeffs = np.polyfit(filtered_magnitudes, filtered_velocities, deg=1)
fit_line = np.poly1d(coeffs)
x_fit = np.linspace(min(filtered_magnitudes), max(filtered_magnitudes), 100)
y_fit = fit_line(x_fit)
ax.plot(x_fit, y_fit, color='red', label='Linear Fit')
ax.set_xlabel('Saccade Magnitude (deg)')

Text(0.5, 0, 'Saccade Magnitude (deg)')